In [1]:
# Step 1. 환경 설정 및 라이브러리 로드
import sys
import os
import torch
import numpy as np
import trimesh
import pickle
import shutil
import subprocess
import cv2
import smplx
import copy
import networkx as nx
import importlib.util
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="kornia")
from scipy.spatial.transform import Rotation as R
from scipy.spatial import KDTree

# === 폴더 경로 설정 ===
from pathlib import Path
import sysconfig

PROJECT_ROOT = Path.cwd()  # main.ipynb가 위치한 프로젝트 루트 디렉토리 기준
INPUT_DIR = PROJECT_ROOT / "input"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR_BASE = "result_output_jupyter"

# === 라이브러리 경로 설정 ===
MODELS_DIR = MODEL_DIR
SITE_PACKAGES_PATH = Path(sysconfig.get_paths()["purelib"])  # 현재 활성화된 가상환경의 site-packages 경로

def add_path(path):
    path = str(path)
    if os.path.exists(path) and path not in sys.path:
        sys.path.insert(0, path)

add_path(SITE_PACKAGES_PATH)
add_path(MODELS_DIR)
add_path(os.path.join(MODELS_DIR, 'smplx'))

# DECA 경로
DECA_ROOT = os.path.join(MODELS_DIR, 'DECA')
add_path(DECA_ROOT)
DECA_DATA_DIR = os.path.join(DECA_ROOT, 'data')
DECA_MODEL_PATH = os.path.join(DECA_DATA_DIR, 'deca_model.tar')

# PIXIE 경로
PIXIE_ROOT = os.path.join(MODELS_DIR, "PIXIE-master")
add_path(PIXIE_ROOT)
PIXIE_DATA_DIR = os.path.join(PIXIE_ROOT, 'data')
PIXIE_MODEL_PATH = os.path.join(PIXIE_DATA_DIR, 'pixie_model.tar')
PIXIE_SMPLX_PATH = os.path.join(PIXIE_DATA_DIR, 'SMPLX_NEUTRAL_2020.npz')

# Import
try:
    from decalib.deca import DECA
    from decalib.utils.config import cfg as deca_cfg
except ImportError:
    try:
        from DECA.decalib.deca import DECA
        from DECA.decalib.utils.config import cfg as deca_cfg
    except ImportError:
        print("[Error] DECA import failed")

try:
    from pixielib.pixie import PIXIE
    from pixielib.datasets.body_datasets import TestData
    from pixielib.utils.config import cfg as pixie_cfg
    from pixielib.utils import util
except ImportError as e:
    print(f"[Error] PIXIE import failed: {e}")

# CPU 강제 설정
device = torch.device('cpu')

#결과 저장 폴더 자동 넘버링
def create_unique_output_dir(base_name):
    if not os.path.exists(base_name): return base_name
    counter = 1
    while True:
        target_dir = f"{base_name}{counter}"
        if not os.path.exists(target_dir): return target_dir
        counter += 1

print(f"=== [System] Running on: {device} (CPU Forced) ===")
print(f"=== [Path] Project Root: {PROJECT_ROOT}")
save_dir = create_unique_output_dir(OUTPUT_DIR_BASE)
os.makedirs(save_dir, exist_ok=True)
print(f"=== [Output] Results will be saved to: {save_dir}")

In [2]:
# Step 2. 함수 로드
# 공통 함수 및 방법론별 함수 로드

# =============================================================================
# 공통 함수 
# =============================================================================   
#3D 메시 저장
def save_obj(mesh, path):
    try:
        mesh.export(path)
        print(f" -> Saved: {os.path.basename(path)}")
    except Exception as e:
        print(f"[Error] Failed to save {path}: {e}")
        
# =============================================================================
# 전신 메시 복원 함수
# =============================================================================
# 전신 메시 생성용 파라미터 추출 및 전처리 
def get_body_parameters(data_dict):
    # 입력된 데이터가 텐서인지 넘파이인지 확인 후 CPU 텐서로 변환
    def to_tensor(val):
        if val is None: return None
        if isinstance(val, torch.Tensor):
            return val.detach().cpu().float() # 무조건 CPU로
        return torch.from_numpy(val).float() # 무조건 CPU로

    # 1. Betas
    betas = data_dict.get('betas')
    if betas is None: betas = data_dict.get('shape')
    
    # 2. Pose (PIXIE 통합 포즈 분리)
    global_orient = data_dict.get('global_orient')
    body_pose = data_dict.get('body_pose')
    jaw_pose = data_dict.get('jaw_pose')
    
    if global_orient is None and 'pose' in data_dict:
        full_pose = to_tensor(data_dict['pose'])
        # SMPL-X 포맷: 0-3(Global), 3-66(Body), 66-69(Jaw)
        if full_pose.shape[-1] >= 3:
            global_orient = full_pose[:, :3]
        if full_pose.shape[-1] >= 66:
            body_pose = full_pose[:, 3:66]
        if full_pose.shape[-1] >= 69:
            jaw_pose = full_pose[:, 66:69]

    # 3. Hands & Transl
    left_hand_pose = data_dict.get('left_hand_pose')
    right_hand_pose = data_dict.get('right_hand_pose')
    
    transl = data_dict.get('transl')
    if transl is None:
        if 'cam' in data_dict:
            # PIXIE cam 파라미터 가져오기
            t_val = data_dict['cam'][:, 1:]
            z_zeros = torch.zeros((t_val.shape[0], 1))
            transl = torch.cat([to_tensor(t_val), z_zeros], dim=1)
        else:
            transl = torch.zeros((1, 3))

    # 안전 장치 
    if global_orient is None: global_orient = torch.zeros((1, 3))
    if body_pose is None: body_pose = torch.zeros((1, 63))
    if jaw_pose is None: jaw_pose = torch.zeros((1, 3))
    if betas is None: betas = torch.zeros((1, 10))
    if left_hand_pose is None: left_hand_pose = torch.zeros((1, 12))
    if right_hand_pose is None: right_hand_pose = torch.zeros((1, 12))

    return {
        'global_orient': to_tensor(global_orient),
        'body_pose': to_tensor(body_pose),
        'left_hand_pose': to_tensor(left_hand_pose),
        'right_hand_pose': to_tensor(right_hand_pose),
        'transl': to_tensor(transl),
        'betas': to_tensor(betas),
        'jaw_pose': to_tensor(jaw_pose)
    }

# =============================================================================
# Method_A, B 함수 
# =============================================================================
#단면 버텍스 탐색 
def find_boundary_loop_clean_py(mesh, plane_origin, plane_normal, tolerance=0.05):
    unique_edges = mesh.edges_sorted
    unique, counts = np.unique(unique_edges, return_counts=True, axis=0)
    boundary_edges = unique[counts == 1]
    
    if len(boundary_edges) == 0: return None

    verts = mesh.vertices
    dists = np.dot(verts - plane_origin, plane_normal)
    #절단면 평면 근처의 엣지들만 필터링 
    filtered_edges = []
    for u, v in boundary_edges:
        if abs(dists[u]) < tolerance or abs(dists[v]) < tolerance:
            filtered_edges.append((u, v))
            
    if not filtered_edges: filtered_edges = boundary_edges

    #그래프를 만들어 순환하는 루프 찾음
    G = nx.Graph()
    G.add_edges_from(filtered_edges)
    try:
        cycles = nx.cycle_basis(G)
        if cycles: return max(cycles, key=len)#가장 큰 구멍 반환 
        return list(max(nx.connected_components(G), key=len))
    except: return None

#스무딩 
def local_laplacian_smooth_py(mesh, target_indices, iterations=10, lambda_val=0.5):
    if len(target_indices) == 0: return mesh
    neighbors_list = mesh.vertex_neighbors
    verts = mesh.vertices.copy()
    for _ in range(iterations):
        updates = np.zeros((len(target_indices), 3))
        for i, idx in enumerate(target_indices):
            n_idxs = neighbors_list[idx]
            if len(n_idxs) > 0:
                avg = np.mean(verts[n_idxs], axis=0)
                #현재 위치를 이웃들의 평균 위치쪽으로 조금 이동
                updates[i] = verts[idx] + lambda_val * (avg - verts[idx])
        verts[target_indices] = updates
    mesh.vertices = verts
    return mesh

#목 버텍스 회전 방향 판별 
def check_winding_order_py(vertices, center, normal=None):
    if normal is None:
        centered = vertices - center
        cov = np.cov(centered.T)
        evals, evecs = np.linalg.eigh(cov)
        normal = evecs[:, 0] 
        if normal[1] < 0: normal = -normal

    tangent = np.cross(normal, [0, 0, 1])
    if np.linalg.norm(tangent) < 1e-6: tangent = np.cross(normal, [1, 0, 0])
    tangent = tangent / np.linalg.norm(tangent)
    bitangent = np.cross(normal, tangent)

    v_2d = []
    for v in vertices:
        vec = v - center
        v_2d.append([np.dot(vec, tangent), np.dot(vec, bitangent)])
    v_2d = np.array(v_2d)

    cross_sum = 0
    for i in range(len(v_2d)):
        curr_v = v_2d[i]
        next_v = v_2d[(i + 1) % len(v_2d)]
        cross_sum += (curr_v[0] * next_v[1] - curr_v[1] * next_v[0])
    
    return cross_sum

#전신 메시 목 단면 버텍스를 얼굴 메시 목 버텍스 개수에 맞춰 재배열
def resample_curve_exact_py(vertices, target_count):
    if len(vertices) < 2: return vertices
    diffs = np.diff(vertices, axis=0)
    diffs = np.vstack([diffs, vertices[0] - vertices[-1]])
    dists = np.linalg.norm(diffs, axis=1)
    cum_dist = np.concatenate(([0], np.cumsum(dists)))
    total_length = cum_dist[-1]
    target_dists = np.linspace(0, total_length, target_count, endpoint=False)
    new_verts = []
    for d in target_dists:
        idx = np.searchsorted(cum_dist, d) - 1
        if idx < 0: idx = 0
        segment_len = dists[idx]
        t = (d - cum_dist[idx]) / segment_len if segment_len > 0 else 0.0
        p_start = vertices[idx]
        p_end = vertices[(idx + 1) % len(vertices)]
        p_new = p_start * (1 - t) + p_end * t
        new_verts.append(p_new)
    return np.array(new_verts)
# =============================================================================
# Method_C 함수 
# =============================================================================
#다차원 텐서 2D 변환 
def flatten_tensor(tensor):
    if tensor is None: return None
    return tensor.view(tensor.shape[0], -1)

#회전 행렬 축-각 변환
def convert_to_axis_angle(pose_tensor):
    """Rotation Matrix(9) -> Axis-Angle(3)"""
    if pose_tensor is None: return None
    
    pose_np = pose_tensor.detach().cpu().numpy()
    batch_size = pose_np.shape[0]
    dim = pose_np.shape[1]
    
    if dim % 9 == 0 and dim > 0:
        num_joints = dim // 9
        reshaped = pose_np.reshape(batch_size * num_joints, 3, 3)
        rot = R.from_matrix(reshaped)
        axis_angle = rot.as_rotvec() 
        final_np = axis_angle.reshape(batch_size, num_joints * 3)
        print(f"   [Info] Converted RotMatrix to AxisAngle: {dim} -> {final_np.shape[1]}")
        return torch.from_numpy(final_np).float().to(device)
        
    return pose_tensor.to(device)

#단일 회전 정보 정규화
def ensure_axis_angle_single(pose_data):
    if isinstance(pose_data, torch.Tensor):
        data_np = pose_data.detach().cpu().numpy()
    else:
        data_np = pose_data
    data_np = np.squeeze(data_np) 
    if data_np.shape == (3, 3):
        vec, _ = cv2.Rodrigues(data_np)
        data_np = vec.flatten()
    elif data_np.size == 3:
        data_np = data_np.flatten()
    else:
        data_np = np.zeros(3, dtype=np.float32)
    return torch.from_numpy(data_np).view(1, 3).float().to(device)

#메시 좌표 동기화 및 위치 보정
def align_generated_to_original(gen_verts, gen_joints, target_joints):
    """
    생성된 메시의 뼈대(gen_joints)를 원본 뼈대(target_joints)에 맞춰
    메시 전체를 회전 및 이동시키는 함수 (SVD 사용)
    """
    # 1. 정렬 기준이 될 관절 인덱스 (움직임이 적은 몸통 위주)
    # SMPL-X: 0(Pelvis), 1(L_Hip), 2(R_Hip), 9(Spine3), 12(Neck)
    common_idxs = [0, 1, 2, 9, 12]
    
    # 데이터 추출 (Batch 차원 제거)
    src_j = gen_joints[0, common_idxs].detach().cpu().numpy()
    tgt_j = target_joints[common_idxs] # numpy array 가정
    
    # 2. 중심점(Centroid) 계산 및 원점 이동
    src_center = np.mean(src_j, axis=0)
    tgt_center = np.mean(tgt_j, axis=0)
    
    src_centered = src_j - src_center
    tgt_centered = tgt_j - tgt_center
    
    # 3. 최적 회전 행렬 계산 (SVD)
    H = np.dot(src_centered.T, tgt_centered)
    U, S, Vt = np.linalg.svd(H)
    R_mat = np.dot(Vt.T, U.T)
    
    # 반사(Reflection) 보정 (Det가 음수면 뒤집힌 것)
    if np.linalg.det(R_mat) < 0:
        Vt[2, :] *= -1
        R_mat = np.dot(Vt.T, U.T)
        
    # 4. 버텍스 전체에 변환 적용
    # 공식: (Points - Src_Center) * Rotation + Tgt_Center
    aligned_verts = np.dot(gen_verts - src_center, R_mat.T) + tgt_center
    
    return aligned_verts
# =============================================================================
# Method_D, E 함수 
# =============================================================================
# 입력 텐서/배열 포맷 표준화
def _to_hw1(x):
    if x is None: return None
    if torch.is_tensor(x):
        t = x.detach().cpu()
        if t.ndim == 4: t = t[0]
        if t.ndim == 3: t = t.permute(1, 2, 0)
        elif t.ndim == 2: t = t.unsqueeze(-1)
        arr = t.numpy().astype(np.float32)
    else:
        arr = np.asarray(x, dtype=np.float32)
        if arr.ndim == 2: arr = arr[..., None]
    if arr.ndim == 2: arr = arr[..., None]
    if arr.shape[2] != 1: arr = arr[..., :1]
    return arr

# DECA 디테일 디코더 입력 차원 검사 
def _get_detail_expected_dim():
    if not hasattr(deca, "D_detail"): return -1
    dd = deca.D_detail
    if hasattr(dd, "l1") and hasattr(dd.l1, "in_features"): return int(dd.l1.in_features)
    for m in dd.modules():
        if isinstance(m, torch.nn.Linear): return int(m.in_features)
    return -1

# 잠재 코드 차원 정렬
def _match_dim(code, expected_dim):
    if code is None: return None
    if code.ndim > 2: code = code.view(code.shape[0], -1)
    if expected_dim <= 0: return code
    cur = int(code.shape[1])
    if cur == expected_dim: return code
    if cur > expected_dim: return code[:, :expected_dim]
    return torch.nn.functional.pad(code, (0, expected_dim - cur))

# DECA 기반 변위 맵 추출 및 마스킹 
def _get_deca_displacement_map_from_path(img_path):
    try: from decalib.datasets import datasets
    except ImportError: from DECA.decalib.datasets import datasets
    
    td = datasets.TestData(img_path, iscrop=True, face_detector="fan", sample_step=1)
    if len(td) == 0:
        images = load_image_to_tensor(img_path).cpu()
    else:
        images = td[0]["image"].unsqueeze(0).to("cpu")

    expected_dim = _get_detail_expected_dim()
    with torch.no_grad():
        candidates = []
        if hasattr(deca, "E_detail"): candidates.append(deca.E_detail(images))
        if hasattr(deca, "detail_encoder"): candidates.append(deca.detail_encoder(images))
        coded = deca.encode(images)
        if isinstance(coded, dict) and "detail" in coded: candidates.append(coded["detail"])

        if not candidates: return np.zeros((target_res, target_res, 1), dtype=np.float32)
        
        detailcode = None
        if expected_dim > 0:
            for c in candidates:
                if c is None: continue
                c2 = c.view(c.shape[0], -1) if c.ndim > 2 else c
                if int(c2.shape[1]) == expected_dim:
                    detailcode = c2; break
        if detailcode is None: detailcode = _match_dim(candidates[0], expected_dim)

        if hasattr(deca, "D_detail"): uv_z_t = deca.D_detail(detailcode)
        elif hasattr(deca, "detail_decoder"): uv_z_t = deca.detail_decoder(detailcode)
        else: return np.zeros((target_res, target_res, 1), dtype=np.float32)

    uv_z = _to_hw1(uv_z_t)

    
    for mk in ["uv_face_eye_mask", "uv_face_mask", "face_eye_mask", "uv_mask"]:
        if hasattr(deca, mk):
            m = getattr(deca, mk)
            m_hw1 = _to_hw1(m)
            if m_hw1 is not None and m_hw1.shape[:2] == uv_z.shape[:2]:
                uv_z = uv_z * m_hw1
                # print(f"    [Info] Applied mask: {mk}")
            break

    return uv_z.astype(np.float32)

# UV 좌표 기반 이중 선형 보간 샘플링 
def _bilinear_sample_hw1(tex_hw1, uv_v2):
    H, W = tex_hw1.shape[0], tex_hw1.shape[1]
    u, v = np.clip(uv_v2[:, 0], 0, 1), np.clip(uv_v2[:, 1], 0, 1)
    x = u * (W - 1); y = (1.0 - v) * (H - 1)
    x0 = np.floor(x).astype(np.int32); y0 = np.floor(y).astype(np.int32)
    x1 = np.clip(x0 + 1, 0, W - 1); y1 = np.clip(y0 + 1, 0, H - 1)
    wx, wy = (x - x0).astype(np.float32), (y - y0).astype(np.float32)
    d0 = tex_hw1[y0, x0, 0] * (1.0 - wx) + tex_hw1[y0, x1, 0] * wx
    d1 = tex_hw1[y1, x0, 0] * (1.0 - wx) + tex_hw1[y1, x1, 0] * wx
    return (d0 * (1.0 - wy) + d1 * wy).astype(np.float32)

# 메시 정점 법선 벡터 계산
def _vertex_normals(V, F):
    n = np.zeros_like(V, dtype=np.float32)
    v0, v1, v2 = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]
    fn = np.cross(v1 - v0, v2 - v0).astype(np.float32)
    for k in range(3): n[F[:, k]] += fn
    norm = np.linalg.norm(n, axis=1, keepdims=True) + 1e-8
    return n / norm

In [5]:
# Step 3. 전신 메시 복원  

# === 폴더 경로 설정 ===
BODY_IMAGE_PATH = str(INPUT_DIR / "SMPL-X" / "body5.jpg")

# =============================================================================
# [Stage 1] Body Generation 
# =============================================================================
import torch
import numpy as np
import os
import trimesh
from pixielib.utils import util
import smplx
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"\n>>> [1/2] Generating Body Mesh using PIXIE from {os.path.basename(BODY_IMAGE_PATH)}...")

# CPU 강제
pixie_cfg.device = "cpu"
pixie_cfg.model.use_tex = False 
pixie_cfg.model.n_shape = 10 
pixie_cfg.model.n_exp = 10 
pixie_cfg.pretrained_modelpath = PIXIE_MODEL_PATH
pixie_cfg.model.smplx_model_path = PIXIE_SMPLX_PATH

# 모델 로드
pixie_model = None
try:
    print(" -> Loading PIXIE Model on CPU...")
    pixie_model = PIXIE(config=pixie_cfg, device='cpu')
    print(" -> [Success] PIXIE Model Loaded.")
except Exception as e:
    print(f" -> [Error] Failed to load PIXIE model: {e}")

if pixie_model:
    # 데이터 로드
    try:
        print(" -> Loading Image Data...")
        
        # iscrop=True 설정
        testdata = TestData(BODY_IMAGE_PATH, iscrop=True, body_detector="rcnn", device='cpu')
        print(f" -> [Info] Detected {len(testdata)} person(s).")
    except Exception as e:
        print(f" -> [Error] Failed to load image data: {e}")
        testdata = []

    if len(testdata) > 0:
        # 데이터 이동 (CPU)
        batch = testdata[0]
        util.move_dict_to_device(batch, 'cpu')
        
        batch["image"] = batch["image"].unsqueeze(0)
        batch["image_hd"] = batch["image_hd"].unsqueeze(0)
        
        data = {"body": batch}
        
        # [Inference]
        print(" -> Running PIXIE Inference...")
        with torch.no_grad():
            param_dict = pixie_model.encode(data, threthold=False, keep_local=True, copy_and_paste=False)
            
        codedict = param_dict["body"]
        
        # [Decoding]
        with torch.no_grad():
            pred_dict = pixie_model.decode(codedict, param_type="body")
        
        # ---------------------------------------------------------------------
        # transformed_vertices 사용
        # ---------------------------------------------------------------------
        verts = pred_dict["transformed_vertices"][0].detach().cpu().numpy()
        faces = pixie_model.smplx.faces_tensor.detach().cpu().numpy()
        
        # ---------------------------------------------------------------------
        # Upright Mesh 저장 및 Global Mesh 설정
        # ---------------------------------------------------------------------
        
        # 1) Upright Mesh 생성 및 저장 
        verts_upright = verts.copy()
        verts_upright[:, 1] = -verts_upright[:, 1]
        verts_upright[:, 2] = -verts_upright[:, 2]
        
        body_mesh_upright = trimesh.Trimesh(vertices=verts_upright, faces=faces, process=False)
        upright_path = os.path.join(save_dir, '0_pixie_body_upright.obj')
        try:
            body_mesh_upright.export(upright_path)
            print(f" -> [Success] Upright Body Mesh Saved: {upright_path}")
        except Exception as e:
            print(f" -> [Error] Failed to write upright file: {e}")

        # 2) Raw Mesh 
        body_mesh_global = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
    
        raw_path = os.path.join(save_dir, '0_pixie_body_raw.obj')
        try:
            body_mesh_global.export(raw_path)
            print(f" -> [Success] Raw Body Mesh Saved (For Methodology): {raw_path}")
        except Exception as e:
            print(f" -> [Error] Failed to write raw file: {e}")

        # ---------------------------------------------------------------------
        # CPU 계산
        # ---------------------------------------------------------------------
        print(" -> Calculating Joints (on CPU)...")
        smplx_model_folder = os.path.join(MODELS_DIR, 'smplx', 'models')
        
        temp_layer = smplx.create(smplx_model_folder, model_type='smplx', gender='neutral').to('cpu')
        j_reg = temp_layer.J_regressor
        
        # body_mesh_global(Raw) 기준으로 관절 계산
        v_tensor = torch.from_numpy(body_mesh_global.vertices).float()
        
        body_joints_global = torch.matmul(j_reg, v_tensor).detach().numpy()
        if len(body_joints_global.shape) == 3: body_joints_global = body_joints_global[0]

        # ---------------------------------------------------------------------
        #  파라미터 추출
        # ---------------------------------------------------------------------
        print(" -> Extracting Parameters...")
        codedict_cpu = {k: v.detach().cpu().float() if isinstance(v, torch.Tensor) else torch.from_numpy(v).float() 
                        for k, v in codedict.items() if v is not None}
        
        body_params_global = get_body_parameters(codedict_cpu)
        print(" -> [Complete] Body preparation finished.")
        
    else:
        print("\n[Warning] No person detected in the image!")
        print(" - Try checking if the image path is correct.")
        print(" - Ensure the person is clearly visible.")
        body_mesh_global = None
else:
    print("[Error] PIXIE model not initialized.")


>>> [1/2] Generating Body Mesh using PIXIE from body5.jpg...
 -> Loading PIXIE Model on CPU...
creating the SMPLX Decoder
 -> [Success] PIXIE Model Loaded.
 -> Loading Image Data...
total 1 images
 -> [Info] Detected 1 person(s).
 -> Running PIXIE Inference...
 -> [Success] Upright Body Mesh Saved: result_output_jupyter\0_pixie_body_upright.obj
 -> [Success] Raw Body Mesh Saved (For Methodology): result_output_jupyter\0_pixie_body_raw.obj
 -> Calculating Joints (on CPU)...
 -> Extracting Parameters...
 -> [Complete] Body preparation finished.


In [6]:
# Step 4. 얼굴 메시 복원

# === 폴더 경로 설정 ===
SOURCE_FACE_IMAGE_PATH = str(INPUT_DIR / "DECA" / "face4.jpg")
DRIVING_FACE_IMAGE_PATH = str(INPUT_DIR / "DECA" / "face1.png")

print(f"\n>>> [2/2] Generating Face Parameters from Two Images...")

# =============================================================================
# [초기화] DECA 모델 설정
# =============================================================================
deca_cfg.model.flame_model_path = os.path.join(DECA_DATA_DIR, 'generic_model.pkl')
deca_cfg.model.use_tex = False
deca_cfg.rasterizer_type = 'pytorch3d'
deca_cfg.pretrained_modelpath = DECA_MODEL_PATH
deca = DECA(config=deca_cfg, device=device)

# =============================================================================
# [전처리] 이미지 로드 및 텐서 변환 함수
# =============================================================================
def load_image_to_tensor(path):
    img = cv2.imread(path)
    if img is None: raise FileNotFoundError(f"이미지를 읽을 수 없습니다: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    # (H, W, C) -> (C, H, W) -> Batch -> Normalize
    return torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).float().to(device) / 255.0

#텐서 변환
img_tensor_source = load_image_to_tensor(SOURCE_FACE_IMAGE_PATH)
img_tensor_driving = load_image_to_tensor(DRIVING_FACE_IMAGE_PATH)

with torch.no_grad():
    codedict_source_base = deca.encode(img_tensor_source) # A: Identity
    codedict_driving_base = deca.encode(img_tensor_driving) # B: Expression & Jaw Pose
    
    # =========================================================================
    # Case 1: Raw Expression (Identity A Original) - 원본 표정
    # =========================================================================

    codedict_raw = copy.deepcopy(codedict_source_base)
    if 'pose' in codedict_raw:
        codedict_raw['pose'][0, :3] = 0.0 
    opdict_raw = deca.decode(codedict_raw)
    deca_lmk_raw_global = opdict_raw['landmarks3d_world'][0].cpu().numpy()
    deca_mesh_raw_global = trimesh.Trimesh(
        vertices=opdict_raw['verts'][0].cpu().numpy(),
        faces=deca.flame.faces_tensor.cpu().numpy(), process=False
    )
    
    save_obj(deca_mesh_raw_global, os.path.join(save_dir, '1_deca_raw.obj'))
    
    # =========================================================================
    # Case 2: Modified Expression (Identity A + Expression B) - 수정 표정
    # =========================================================================

    codedict_mod = copy.deepcopy(codedict_source_base)
    if 'pose' in codedict_mod:
        codedict_mod['pose'][0, :3] = 0.0 
    if 'exp' in codedict_mod and 'exp' in codedict_driving_base:
        codedict_mod['exp'] = codedict_driving_base['exp'].clone()
    if 'pose' in codedict_mod and 'pose' in codedict_driving_base:
        dim_source = codedict_mod['pose'].shape[1]
        dim_driving = codedict_driving_base['pose'].shape[1]
        copy_len = min(3, dim_source - 3, dim_driving - 3)
        if copy_len > 0:
            codedict_mod['pose'][0, 3:3+copy_len] = codedict_driving_base['pose'][0, 3:3+copy_len]
    opdict_mod = deca.decode(codedict_mod)
    deca_lmk_mod_global = opdict_raw['landmarks3d_world'][0].cpu().numpy()
    deca_mesh_mod_global = trimesh.Trimesh(
        vertices=opdict_mod['verts'][0].cpu().numpy(),
        faces=deca.flame.faces_tensor.cpu().numpy(), process=False
    )
    
    save_obj(deca_mesh_mod_global, os.path.join(save_dir, '2_deca_mod.obj'))

print(" -> Face Processing Complete.")
print("    - 1_deca_raw.obj")
print("    - 2_deca_mod.obj")
print(" -> [complente] Parameters saved to 'codedict_raw' and 'codedict_mod'.")

In [7]:
# 5. Method A: PCA 기반 적응형 슬라이싱 및 위상 봉합

import numpy as np
import trimesh
from scipy.spatial.transform import Rotation as R
from scipy.spatial import KDTree
import os

print(f"\n" + "="*60)
print("[Pipeline Start] Method 1: Mesh Stitching (Radar Logic Improved)")
print("="*60)

def stitch_smooth_seamless(deca_mesh, deca_lmk, body_mesh, joints_input, output_dir, suffix=''):
    print(f"\n" + "="*60)
    print(f"[Pipeline Start] Geometry Stitching Process: {suffix.replace('_', ' ').strip().title()}")
    print("="*60)

    # 원본 데이터 보존을 위한 깊은 복사
    joints = joints_input.copy()

    # 원본 데이터 복사
    head_mesh = deca_mesh.copy()
    body_mesh = body_mesh.copy()
    deca_lmk = deca_lmk.copy()

    # 주요 관절 인덱스 정의
    SPINE3_IDX = 9       
    NECK_IDX = 12        
    HEAD_IDX = 15        
    JAW_IDX = 22        

    # 주요 관절 좌표 추출
    neck_joint = joints[NECK_IDX].copy()
    head_joint = joints[HEAD_IDX].copy()
    spine3_joint = joints[SPINE3_IDX].copy() 
    jaw_joint = joints[JAW_IDX].copy()
    neck_joint_orig = neck_joint.copy() # 복구용 원본 저장

    # =============================================================================
    # [step 0] 좌표계 정규화
    # =============================================================================
    print(" [Step 0] Preprocessing: Normalizing coordinates (Translating Neck to Global Origin)...")
    body_mesh.vertices -= neck_joint
    head_joint -= neck_joint
    spine3_joint -= neck_joint
    jaw_joint -= neck_joint
    joints -= neck_joint

    # =============================================================================
    # [step 1] 전신 정렬
    # =============================================================================
    print(" [Step 1] Body Alignment: Compensating for spinal curvature (Y-axis alignment)...")
    current_neck_vec = jaw_joint 
    current_neck_vec = current_neck_vec / np.linalg.norm(current_neck_vec)
    target_vec = np.array([0., 1., 0.]) 
    
    rotation_axis = np.cross(current_neck_vec, target_vec)
    axis_norm = np.linalg.norm(rotation_axis)
    
    rot_mat_body = np.eye(3)
    if axis_norm > 1e-6:
        rotation_axis = rotation_axis / axis_norm
        dot_val = np.clip(np.dot(current_neck_vec, target_vec), -1.0, 1.0)
        angle = np.arccos(dot_val)
        rot_mat_body = R.from_rotvec(rotation_axis * angle).as_matrix()
    
    EXTRA_TILT_DEG = 15.0
    rot_extra = R.from_euler('x', -EXTRA_TILT_DEG, degrees=True).as_matrix()
    final_rot_mat = np.dot(rot_extra, rot_mat_body)

    body_mesh.vertices = np.dot(body_mesh.vertices, final_rot_mat.T)
    head_joint = np.dot(head_joint, final_rot_mat.T) 
    jaw_joint = np.dot(jaw_joint, final_rot_mat.T) 
    joints = np.dot(joints, final_rot_mat.T) 
    neck_length = np.linalg.norm(head_joint)
    
    #body_mesh.export(os.path.join(output_dir, f'0_debug_body_vertical_aligned{suffix}.obj'))

    # =============================================================================
    # [step 2] 전신 메시 처리 및 절단 
    # =============================================================================
    print(" [Step 2] Adaptive Slicing: 1st PCA Filter & 2nd Radial Radar Check...")
    
    chin_bottom_limit = jaw_joint[1] - 0.03
    start_ratio_high = 0.60 
    end_ratio_low = 0.10 
    step_ratio_down = -0.005 
    
    final_cut_height = 0.0
    body_verts_orig = body_mesh.vertices.copy()
    
    # Fallback 높이 상향 조정 
    fallback_height = neck_length * 0.40
    
    # 후보군 분리 저장 
    candidates_step1_pca = []  
    candidates_step2_radar = [] 

    for r in np.arange(start_ratio_high, end_ratio_low, step_ratio_down):
        h = neck_length * r
        if h > chin_bottom_limit: continue

        mask = (body_verts_orig[:, 1] > h - 0.002) & (body_verts_orig[:, 1] < h + 0.002)
        pts = body_verts_orig[mask]
        
        if len(pts) > 10:
            # 1. PCA 분석
            pts_2d = pts[:, [0, 2]]
            mean = np.mean(pts_2d, axis=0)
            centered = pts_2d - mean
            cov = np.cov(centered.T)
            eigenvalues, _ = np.linalg.eigh(cov)
            
            min_eig = np.min(eigenvalues)
            max_eig = np.max(eigenvalues)
            
            circularity = 1.0 
            if max_eig > 0:
                axis_ratio = np.sqrt(min_eig) / np.sqrt(max_eig)
                circularity = 1.0 - axis_ratio

            current_radius = np.mean(np.linalg.norm(centered, axis=1))

            # PCA 검사 
            if circularity < 0.45 and current_radius < 0.12:
                candidate_info = {
                    'h': h,
                    'radius': current_radius,
                    'circularity': circularity,
                    'ratio': r
                }
                candidates_step1_pca.append(candidate_info)

                # 방사형 레이더 검사 
                # 중심으로부터 각 점까지의 거리 분포 확인
                radial_dists = np.linalg.norm(centered, axis=1)
                
                if len(radial_dists) > 0:
                    radial_median = np.median(radial_dists)
                    radial_max = np.max(radial_dists)
                    
                    radar_ratio = radial_max / (radial_median + 1e-9)
                    radar_diff = radial_max - radial_median
                    
                    if radar_ratio < 1.15 and radar_diff < 0.03:
                        candidate_info['radial_max'] = radial_max
                        candidates_step2_radar.append(candidate_info)

    # 턱 돌출 감지 
    # 2차 검증된 단면들을 아래(Neck) -> 위(Chin) 순서로 정렬 후, 튀어나오는 부분 감지
    if len(candidates_step2_radar) > 1:
        # h 오름차순 정렬 (아래 -> 위)
        candidates_step2_radar.sort(key=lambda x: x['h'])
        
        cutoff_index = -1
        PROTRUSION_THRESHOLD = 0.013
        
        for i in range(1, len(candidates_step2_radar)):
            prev_slice = candidates_step2_radar[i-1]
            curr_slice = candidates_step2_radar[i]
            
            diff = curr_slice['radial_max'] - prev_slice['radial_max']
            
            if diff > PROTRUSION_THRESHOLD:
                print(f"    [Filter] Chin protrusion detected at h={curr_slice['h']:.4f} (Diff: {diff:.4f}). Excluding upper candidates.")
                cutoff_index = i
                break
        
        # 턱이 감지된 경우, 해당 단면부터 그 위쪽은 모두 후보에서 제외
        if cutoff_index != -1:
            candidates_step2_radar = candidates_step2_radar[:cutoff_index]

    # 최적 높이 선정 
    if len(candidates_step2_radar) > 0:
        print(f"    [Selection] Priority 1: Radar Passed & Protrusion Filtered ({len(candidates_step2_radar)} found).")
        best_candidate = min(candidates_step2_radar, key=lambda x: x['radius'])
        final_cut_height = best_candidate['h']
        
    elif len(candidates_step1_pca) > 0:
        print(f"    [Selection] Priority 2: Radar failed. Fallback to PCA candidates ({len(candidates_step1_pca)} found).")
        best_candidate = min(candidates_step1_pca, key=lambda x: x['radius'])
        final_cut_height = best_candidate['h']
        
    else:
        print(f"    [Warning] No valid candidates. Using Safe Fallback Height.")
        safe_fallback = min(fallback_height, chin_bottom_limit - 0.01)
        final_cut_height = safe_fallback

    # 그래프 탐색을 통한 선택적 헤드 제거 
    verts = body_mesh.vertices
    seed_mask = (verts[:, 1] > final_cut_height) & \
                (verts[:, 1] < final_cut_height + 0.02) & \
                ((verts[:, 0]**2 + verts[:, 2]**2) < 0.06**2)
    
    seed_indices = np.where(seed_mask)[0]
    
    if len(seed_indices) > 0:
        remove_mask = np.zeros(len(verts), dtype=bool)
        remove_mask[seed_indices] = True
        neighbors = body_mesh.vertex_neighbors
        stack = list(seed_indices)
        while stack:
            curr_idx = stack.pop()
            valid_next = [n for n in neighbors[curr_idx] if not remove_mask[n] and verts[n, 1] > final_cut_height]
            for n in valid_next: remove_mask[n] = True; stack.append(n)
        
        faces_to_remove = remove_mask[body_mesh.faces].any(axis=1)
        body_mesh.update_faces(~faces_to_remove)
        body_mesh.remove_unreferenced_vertices()

        # 잔여 파편 제거 
        components = body_mesh.split(only_watertight=False)
        keepers = []
        for comp in components:
            if len(comp.vertices) > 2000: keepers.append(comp); continue
            center = comp.centroid
            if center[1] < final_cut_height: keepers.append(comp); continue
            if (center[0]**2 + center[2]**2) > 0.15**2: keepers.append(comp); continue 
            extents = comp.extents
            if np.any(extents < 0.001): keepers.append(comp); continue
            if (np.min(extents) / np.max(extents)) > 0.5: continue 
            keepers.append(comp)
        
        if len(keepers) < len(components):
            body_mesh = trimesh.util.concatenate(keepers)
    else:
        print("[Warning] Fallback slicing used.")
        mask_remove = (verts[:, 1] > final_cut_height)
        body_mesh.update_faces(~mask_remove[body_mesh.faces].any(axis=1))
        body_mesh.remove_unreferenced_vertices()
        
    #body_mesh.export(os.path.join(output_dir, f'3_body_cut_dynamic{suffix}.obj'))

    # =============================================================================
    # [step 3] 얼굴 메시 처리
    # =============================================================================
    print(" [Step 3] Face Processing: Aligning and slicing DECA head mesh...")

    l_eye = np.mean(deca_lmk[36:42], axis=0)
    r_eye = np.mean(deca_lmk[42:48], axis=0)
    pivot = (l_eye + r_eye) / 2.0
    head_mesh.vertices -= pivot
    deca_lmk -= pivot

    verts_base = head_mesh.vertices.copy()
    mesh_min_y = np.min(verts_base[:, 1])
    mesh_chin_y = deca_lmk[8, 1]
    y_steps = np.linspace(mesh_min_y + 0.005, mesh_chin_y, 50)

    best_fallback = None
    min_circularity = float('inf')

    for y in y_steps:
        mask = (verts_base[:, 1] > y - 0.002) & (verts_base[:, 1] < y + 0.002)
        pts = verts_base[mask]
        if len(pts) > 10:
            x_rng = np.ptp(pts[:, 0]); z_rng = np.ptp(pts[:, 2])
            if x_rng > 0:
                ratio = z_rng / x_rng
                circularity = abs(1.0 - ratio)
                current_radius = (x_rng + z_rng) / 4.0
                center_xz = np.array([np.mean(pts[:,0]), np.mean(pts[:,2])])
                current_info = {'radius': current_radius, 'y': y, 'center': center_xz, 'circularity': circularity}
                if circularity < min_circularity:
                    min_circularity = circularity
                    best_fallback = current_info

    if best_fallback is None:
        final_y = mesh_min_y + 0.001
    else:
        final_y = best_fallback['y']
    
    head_mesh = head_mesh.slice_plane(plane_origin=[0, final_y, 0], plane_normal=[0, 1, 0])
    face_boundary_loop = find_boundary_loop_clean_py(head_mesh, [0, final_y, 0], [0, -1, 0])
    face_bottom_indices = []
    if face_boundary_loop is not None:
        face_bottom_indices = face_boundary_loop

    #head_mesh.export(os.path.join(output_dir, f'debug_4c_face_extruded{suffix}.obj'))
    
    # =============================================================================
    # [step 4] 병합 및 토폴로지 연결
    # =============================================================================
    print(" [Step 4] Mesh Merging: Synchronizing local coordinate systems (Body <-> Face)...")

    body_boundary_loop = find_boundary_loop_clean_py(body_mesh, [0, final_cut_height, 0], [0, 1, 0])
    
    if body_boundary_loop is not None and len(face_bottom_indices) > 0:

        # 1) 좌표계 동기화
        if len(joints) >= 25:
            # [Step 4-0] Scaling
            b_l_eye_pos = joints[23]
            b_r_eye_pos = joints[24]
            body_eye_dist = np.linalg.norm(b_r_eye_pos - b_l_eye_pos)

            d_l_eye_pos = np.mean(deca_lmk[36:42], axis=0)
            d_r_eye_pos = np.mean(deca_lmk[42:48], axis=0)
            deca_eye_dist = np.linalg.norm(d_r_eye_pos - d_l_eye_pos)

            if deca_eye_dist > 0:
                scale_factor = body_eye_dist / deca_eye_dist
                print(f"    [Scaling] Applying scale factor: {scale_factor:.4f}")
                head_mesh.vertices *= scale_factor
                deca_lmk *= scale_factor
            else:
                print("    [Warning] DECA eye distance is zero. Skipping scaling.")

            # Body Frame
            b_l_eye = joints[23]
            b_r_eye = joints[24]
            b_neck = joints[NECK_IDX] 
            b_head = joints[HEAD_IDX] 
            b_vec_x = b_r_eye - b_l_eye
            b_vec_x /= np.linalg.norm(b_vec_x)
            b_vec_y_raw = b_head - b_neck 
            b_vec_y_raw /= np.linalg.norm(b_vec_y_raw)
            b_vec_z = np.cross(b_vec_x, b_vec_y_raw)
            b_vec_z /= np.linalg.norm(b_vec_z)
            b_vec_z = -b_vec_z 
            b_vec_x = -b_vec_x
            b_vec_y = np.cross(b_vec_z, b_vec_x)
            b_vec_y /= np.linalg.norm(b_vec_y)
            mat_body = np.stack([b_vec_x, b_vec_y, b_vec_z], axis=1)
            
            # Face Frame
            d_l_eye = np.mean(deca_lmk[36:42], axis=0)
            d_r_eye = np.mean(deca_lmk[42:48], axis=0)
            d_pivot = (d_l_eye + d_r_eye) / 2.0
            
            d_vec_x = d_r_eye - d_l_eye
            d_vec_x /= np.linalg.norm(d_vec_x)
            d_vec_y_raw = np.array([0.0, 1.0, 0.0])
            d_vec_z = np.cross(d_vec_x, d_vec_y_raw)
            d_vec_z /= np.linalg.norm(d_vec_z)
            d_vec_y = np.cross(d_vec_z, d_vec_x)
            d_vec_y /= np.linalg.norm(d_vec_y)
            mat_deca = np.stack([d_vec_x, d_vec_y, d_vec_z], axis=1)

            rot_matrix = np.dot(mat_body, mat_deca.T)
            head_mesh.vertices = np.dot(head_mesh.vertices, rot_matrix.T)
            print(f"    [Alignment] Corrected 180-deg Flip & 45-deg Tilt using Spine Vector.")

        # 2) 위치 정렬 
        b_verts = body_mesh.vertices[body_boundary_loop]
        f_verts = head_mesh.vertices[face_bottom_indices]

        def get_bbox_center_xz(pts):
            # 점들의 기하학적 중심(Bounding Box Center)을 반환
            min_xyz = np.min(pts, axis=0)
            max_xyz = np.max(pts, axis=0)
            center_x = (min_xyz[0] + max_xyz[0]) / 2.0
            center_z = (min_xyz[2] + max_xyz[2]) / 2.0
            return np.array([center_x, center_z])

        body_neck_center_xz = get_bbox_center_xz(b_verts)
        face_center_xz = get_bbox_center_xz(f_verts)
        

        target_pos = np.array([body_neck_center_xz[0], final_cut_height, body_neck_center_xz[1]])
        

        face_mean_y = np.mean(f_verts[:, 1])
        face_current_center = np.array([face_center_xz[0], face_mean_y, face_center_xz[1]])
        
        translation = target_pos - face_current_center
        head_mesh.vertices += translation 

        #head_mesh.export(os.path.join(output_dir, f'4_debug_aligned_face_only{suffix}.obj'))
        #trimesh.Scene([body_mesh, head_mesh]).export(os.path.join(output_dir, f'4_debug_alignment_scene{suffix}.obj'))

        # 3) 토폴로지 매칭
        head_loop_verts = head_mesh.vertices[face_bottom_indices]
        body_loop_verts = body_mesh.vertices[body_boundary_loop]

        wind_head = check_winding_order_py(head_loop_verts, np.mean(head_loop_verts, axis=0))
        wind_body = check_winding_order_py(body_loop_verts, np.mean(body_loop_verts, axis=0))
        
        if (wind_head * wind_body) < 0:
            body_boundary_loop = body_boundary_loop[::-1]
            body_loop_verts = body_mesh.vertices[body_boundary_loop]

        target_count = len(face_bottom_indices)
        original_count = len(body_boundary_loop)
        new_body_loop_verts = resample_curve_exact_py(body_loop_verts, target_count)

        h0 = head_loop_verts[0]
        dists = np.linalg.norm(new_body_loop_verts - h0, axis=1)
        best_roll = np.argmin(dists)
        new_body_loop_verts = np.roll(new_body_loop_verts, -best_roll, axis=0)

        head_r = np.mean(np.linalg.norm(head_loop_verts[:, [0,2]] - target_pos[[0,2]], axis=1))
        body_r = np.mean(np.linalg.norm(new_body_loop_verts[:, [0,2]] - target_pos[[0,2]], axis=1))

        if body_r < head_r:
            print("    [Info] Body neck smaller. Flaring to match head...")
            tree = KDTree(head_loop_verts)
            _, indices = tree.query(new_body_loop_verts)
            new_body_loop_verts[:, 0] = head_loop_verts[indices][:, 0]
            new_body_loop_verts[:, 2] = head_loop_verts[indices][:, 2]

        start_adapter_idx = len(body_mesh.vertices)
        body_mesh.vertices = np.vstack([body_mesh.vertices, new_body_loop_verts])

        adapter_indices = np.arange(start_adapter_idx, start_adapter_idx + target_count)
        adapter_faces = []
        bridge_faces = []
        face_idx_offset = len(body_mesh.vertices) 
        
        for i in range(target_count):
            curr_adapter = adapter_indices[i]
            next_adapter = adapter_indices[(i + 1) % target_count]
            
            t = (i + best_roll) / target_count 
            idx_orig = int(t * original_count) % original_count
            curr_orig = body_boundary_loop[idx_orig]
            
            t_next = (i + 1 + best_roll) / target_count
            idx_orig_next = int(t_next * original_count) % original_count
            next_orig = body_boundary_loop[idx_orig_next]
            
            adapter_faces.append([curr_adapter, curr_orig, next_adapter])
            
            if curr_orig != next_orig:
                step = 1
                if idx_orig_next < idx_orig: step = (original_count - idx_orig) + idx_orig_next
                else: step = idx_orig_next - idx_orig
                temp_curr = idx_orig
                for k in range(step):
                    actual_curr = body_boundary_loop[temp_curr % original_count]
                    actual_next = body_boundary_loop[(temp_curr + 1) % original_count]
                    adapter_faces.append([next_adapter, actual_curr, actual_next])
                    temp_curr += 1

            b_curr = adapter_indices[i]
            b_next = adapter_indices[(i + 1) % target_count]
            f_curr = face_bottom_indices[i] + face_idx_offset
            f_next = face_bottom_indices[(i + 1) % target_count] + face_idx_offset
            
            bridge_faces.append([b_curr, f_curr, b_next])
            bridge_faces.append([f_curr, f_next, b_next])

        combined_verts = np.vstack([body_mesh.vertices, head_mesh.vertices])
        deca_faces_shifted = head_mesh.faces + face_idx_offset
        combined_faces = np.vstack([body_mesh.faces, deca_faces_shifted, adapter_faces, bridge_faces])
        merged_mesh = trimesh.Trimesh(vertices=combined_verts, faces=combined_faces, process=False)
        merged_mesh.merge_vertices(merge_tex=True, merge_norm=True)
        
    # =============================================================================
    # [step 5] 경계선 스무딩
    # =============================================================================
        print(" [Step 5] Seam Smoothing: Applying 5-stage Laplacian smoothing...")
        m_verts = merged_mesh.vertices
        dist_sq_to_axis = (m_verts[:, 0] - body_neck_center_xz[0])**2 + (m_verts[:, 2] - body_neck_center_xz[1])**2
        radius_limit_sq = 0.12**2 

        y_coords = m_verts[:, 1]
        h = final_cut_height
        chest_limit = h - 0.03

        mask1 = (y_coords > h - 0.005) & (y_coords < h + 0.005) & (dist_sq_to_axis < radius_limit_sq)
        zone1 = np.where(mask1)[0]
        if len(zone1) > 0: merged_mesh = local_laplacian_smooth_py(merged_mesh, zone1, iterations=40, lambda_val=0.7)

        mask2 = (y_coords > h - 0.015) & (y_coords < h + 0.015) & (dist_sq_to_axis < radius_limit_sq)
        zone2 = np.where(mask2)[0]
        if len(zone2) > 0: merged_mesh = local_laplacian_smooth_py(merged_mesh, zone2, iterations=30, lambda_val=0.6)

        mask3 = (y_coords > h - 0.030) & (y_coords < h + 0.030) & (dist_sq_to_axis < radius_limit_sq) & (y_coords > chest_limit)
        zone3 = np.where(mask3)[0]
        if len(zone3) > 0: merged_mesh = local_laplacian_smooth_py(merged_mesh, zone3, iterations=20, lambda_val=0.5)

        mask4 = (y_coords > h - 0.050) & (y_coords < h + 0.050) & (dist_sq_to_axis < radius_limit_sq) & (y_coords > chest_limit)
        zone4 = np.where(mask4)[0]
        if len(zone4) > 0: merged_mesh = local_laplacian_smooth_py(merged_mesh, zone4, iterations=15, lambda_val=0.3)

        mask5 = (y_coords > h - 0.080) & (y_coords < h + 0.00) & (dist_sq_to_axis < radius_limit_sq) & (y_coords > chest_limit)
        zone5 = np.where(mask5)[0]
        if len(zone5) > 0: merged_mesh = local_laplacian_smooth_py(merged_mesh, zone5, iterations=10, lambda_val=0.2)
            
        merged_mesh.fix_normals()

    else:
        print("[Error] Failed to stitch.")
        combined_verts = np.vstack([body_mesh.vertices, head_mesh.vertices])
        combined_faces = np.vstack([body_mesh.faces, head_mesh.faces + len(body_mesh.vertices)])
        merged_mesh = trimesh.Trimesh(vertices=combined_verts, faces=combined_faces, process=False)
    
    # =============================================================================
    # [step 6] 최종 복구
    # =============================================================================
    print(" [Step 6] Pose Restoration: Inverse transformation to original global coordinates...")
    merged_mesh.vertices = np.dot(merged_mesh.vertices, final_rot_mat) 
    merged_mesh.vertices += neck_joint_orig

    print(" [Post-Step] Applying Upright Correction for Saving...")
    rot_upright = R.from_euler('x', 180, degrees=True).as_matrix()
    merged_mesh.vertices = np.dot(merged_mesh.vertices, rot_upright.T)

    final_name = f'Method_A{suffix}.obj'
    final_path = os.path.join(output_dir, final_name)
    merged_mesh.export(final_path)
    print(f"    -> [Success] Artifact Saved: {final_name}")

# =============================================================================
# 메인 실행
# =============================================================================

if 'body_mesh_global' in globals() and body_mesh_global is not None:
    print("\n>>> [Exec] Generating Result 1: Original Expression")
    stitch_smooth_seamless(
        deca_mesh_raw_global.copy(), 
        deca_lmk_raw_global.copy(), 
        body_mesh_global.copy(), 
        body_joints_global.copy(), 
        save_dir, 
        suffix="_original"
    )
    
    print("\n>>> [Exec] Generating Result 1: Modified Expression")
    stitch_smooth_seamless(
        deca_mesh_mod_global.copy(), 
        deca_lmk_mod_global.copy(), 
        body_mesh_global.copy(), 
        body_joints_global.copy(), 
        save_dir, 
        suffix="_modified"
    )
else:
    print("[Error] Body Mesh Global variable not found.")


[Pipeline Start] Method 1: Mesh Stitching (Radar Logic Improved)

>>> [Exec] Generating Result 1: Original Expression

[Pipeline Start] Geometry Stitching Process: Original
 [Step 0] Preprocessing: Normalizing coordinates (Translating Neck to Global Origin)...
 [Step 1] Body Alignment: Compensating for spinal curvature (Y-axis alignment)...
 [Step 2] Adaptive Slicing: 1st PCA Filter & 2nd Radial Radar Check...
    [Warning] No valid candidates. Using Safe Fallback Height.
 [Step 3] Face Processing: Aligning and slicing DECA head mesh...
 [Step 4] Mesh Merging: Synchronizing local coordinate systems (Body <-> Face)...
    [Scaling] Applying scale factor: 0.9794
    [Alignment] Corrected 180-deg Flip & 45-deg Tilt using Spine Vector.
    [Info] Body neck smaller. Flaring to match head...
 [Step 5] Seam Smoothing: Applying 5-stage Laplacian smoothing...
 [Step 6] Pose Restoration: Inverse transformation to original global coordinates...
 [Post-Step] Applying Upright Correction for Saving

In [8]:
# 6. Method B: PCA 기반 적응형 슬라이싱 및 하모닉 메시 페어링

import numpy as np
import trimesh
from scipy.spatial.transform import Rotation as R
from scipy.spatial import KDTree
import scipy.sparse as sp 
import os
import copy

print(f"\n" + "="*80)
print("[Pipeline Start] Integrated Hybrid Solution")
print("="*80)

# =============================================================================
# Helper Function: Zone-Based Smoothing 
# =============================================================================
def smooth_neck_zone_based(mesh, cut_height, center_pos):
    
    verts = mesh.vertices.copy()
    n_verts = len(verts)
    
    # 중심축 거리 계산
    rel_x = verts[:, 0] - center_pos[0]
    rel_z = verts[:, 2] - center_pos[2]
    dist_sq = rel_x**2 + rel_z**2
    
    # 높이 차이 계산 (스무딩 영역 설정)
    diff_y = verts[:, 1] - cut_height
    radius_limit_sq = 0.10**2 
    is_inside_radius = dist_sq < radius_limit_sq
    # [Method 2] 스무딩 범위
    is_in_band = (diff_y > -0.03) & (diff_y < 0.08)

    # 최종 마스크
    active_mask = is_inside_radius & is_in_band
    active_indices = np.where(active_mask)[0]

    if len(active_indices) == 0: return mesh

    print(f"    -> Smoothing Zone: {len(active_indices)} vertices.")
    print("        (Range: -8cm to +1.5cm relative to seam, Radius: 12cm)")

    # Matrix Setup (Bi-Laplacian)
    edges = mesh.edges_unique
    row = np.concatenate([edges[:, 0], edges[:, 1]])
    col = np.concatenate([edges[:, 1], edges[:, 0]])
    data = np.ones(len(row))
    A = sp.csr_matrix((data, (row, col)), shape=(n_verts, n_verts))
    degree = np.array(A.sum(axis=1)).flatten()
    degree[degree < 1e-6] = 1.0
    D_inv = sp.diags(1.0 / degree)
    L = sp.eye(n_verts) - D_inv.dot(A)

    current_verts = verts

    print("    [Pass 1] Ironing Zipper Seams...")
    for _ in range(20):
        delta = -L.dot(current_verts)
        current_verts[active_indices] += 0.8 * delta[active_indices]

    print("    [Pass 2] Creating Seamless Curve...")
    for _ in range(50): 
        delta = L.dot(current_verts)
        bi_delta = L.dot(delta)
        current_verts[active_indices] += -0.3 * bi_delta[active_indices]

    mesh.vertices = current_verts
    return mesh

# -----------------------------------------------------------------------------
# [Main Pipeline]
# -----------------------------------------------------------------------------
def stitch_hybrid_final(deca_mesh_in, deca_lmk_in, body_mesh_in, joints_input, output_dir, suffix=''):
    print(f"\n" + "="*60)
    print(f"[Pipeline Start] Geometry Stitching Process: {suffix.replace('_', ' ').strip().title()}")
    print("="*60)

    # 원본 데이터 보존을 위한 깊은 복사
    joints = joints_input.copy()

    # 원본 데이터 복사
    head_mesh = deca_mesh_in.copy()
    body_mesh = body_mesh_in.copy()
    deca_lmk = deca_lmk_in.copy()

    # 주요 관절 인덱스 정의
    SPINE3_IDX = 9       
    NECK_IDX = 12        
    HEAD_IDX = 15        
    JAW_IDX = 22        

    # 주요 관절 좌표 추출
    neck_joint = joints[NECK_IDX].copy()
    head_joint = joints[HEAD_IDX].copy()
    spine3_joint = joints[SPINE3_IDX].copy() 
    jaw_joint = joints[JAW_IDX].copy()
    neck_joint_orig = neck_joint.copy() # 복구용 원본 저장

    # =============================================================================
    # [Step 0] 좌표계 정규화 
    # =============================================================================
    print(" [Step 0] Preprocessing: Normalizing coordinates (Translating Neck to Global Origin)...")
    body_mesh.vertices -= neck_joint
    head_joint -= neck_joint
    spine3_joint -= neck_joint
    jaw_joint -= neck_joint
    joints -= neck_joint

    # =============================================================================
    # [Step 1] 전신 정렬 
    # =============================================================================
    print(" [Step 1] Body Alignment: Compensating for spinal curvature (Y-axis alignment)...")
    current_neck_vec = jaw_joint 
    current_neck_vec = current_neck_vec / np.linalg.norm(current_neck_vec)
    target_vec = np.array([0., 1., 0.]) 
    
    rotation_axis = np.cross(current_neck_vec, target_vec)
    axis_norm = np.linalg.norm(rotation_axis)
    
    rot_mat_body = np.eye(3)
    if axis_norm > 1e-6:
        rotation_axis = rotation_axis / axis_norm
        dot_val = np.clip(np.dot(current_neck_vec, target_vec), -1.0, 1.0)
        angle = np.arccos(dot_val)
        rot_mat_body = R.from_rotvec(rotation_axis * angle).as_matrix()
    
    EXTRA_TILT_DEG = 15.0
    rot_extra = R.from_euler('x', -EXTRA_TILT_DEG, degrees=True).as_matrix()
    final_rot_mat = np.dot(rot_extra, rot_mat_body)

    body_mesh.vertices = np.dot(body_mesh.vertices, final_rot_mat.T)
    head_joint = np.dot(head_joint, final_rot_mat.T) 
    jaw_joint = np.dot(jaw_joint, final_rot_mat.T) 
    joints = np.dot(joints, final_rot_mat.T) 
    neck_length = np.linalg.norm(head_joint)
    
    #body_mesh.export(os.path.join(output_dir, f'0_debug_body_vertical_aligned{suffix}.obj'))

    # =============================================================================
    # [Step 2] 전신 메시 처리 및 절단
    # =============================================================================
    print(" [Step 2] Adaptive Slicing: 1st PCA Filter & 2nd Radial Radar Check (Method 1 Logic)...")
    
    chin_bottom_limit = jaw_joint[1] - 0.03
    start_ratio_high = 0.60 
    end_ratio_low = 0.10 
    step_ratio_down = -0.005 
    
    final_cut_height = 0.0
    body_verts_orig = body_mesh.vertices.copy()
    
    # Fallback 높이 상향 조정 
    fallback_height = neck_length * 0.40
    
    # 후보군 분리 저장 
    candidates_step1_pca = []  
    candidates_step2_radar = [] 

    for r in np.arange(start_ratio_high, end_ratio_low, step_ratio_down):
        h = neck_length * r
        if h > chin_bottom_limit: continue

        mask = (body_verts_orig[:, 1] > h - 0.002) & (body_verts_orig[:, 1] < h + 0.002)
        pts = body_verts_orig[mask]
        
        if len(pts) > 10:
            # 1. PCA 분석
            pts_2d = pts[:, [0, 2]]
            mean = np.mean(pts_2d, axis=0)
            centered = pts_2d - mean
            cov = np.cov(centered.T)
            eigenvalues, _ = np.linalg.eigh(cov)
            
            min_eig = np.min(eigenvalues)
            max_eig = np.max(eigenvalues)
            
            circularity = 1.0 
            if max_eig > 0:
                axis_ratio = np.sqrt(min_eig) / np.sqrt(max_eig)
                circularity = 1.0 - axis_ratio

            current_radius = np.mean(np.linalg.norm(centered, axis=1))

            # PCA 검사 
            if circularity < 0.45 and current_radius < 0.12:
                candidate_info = {
                    'h': h,
                    'radius': current_radius,
                    'circularity': circularity,
                    'ratio': r
                }
                candidates_step1_pca.append(candidate_info)

                # 방사형 레이더 검사 
                # 중심으로부터 각 점까지의 거리 분포 확인
                radial_dists = np.linalg.norm(centered, axis=1)
                
                if len(radial_dists) > 0:
                    radial_median = np.median(radial_dists)
                    radial_max = np.max(radial_dists)
                    
                    radar_ratio = radial_max / (radial_median + 1e-9)
                    radar_diff = radial_max - radial_median
                    
                    if radar_ratio < 1.15 and radar_diff < 0.03:
                        candidate_info['radial_max'] = radial_max
                        candidates_step2_radar.append(candidate_info)

    #턱 돌출 감지
    if len(candidates_step2_radar) > 1:
        # h 오름차순 정렬 (아래 -> 위)
        candidates_step2_radar.sort(key=lambda x: x['h'])
        
        cutoff_index = -1
        # 임계값 이상 급격히 튀어나오면 턱으로 간주
        PROTRUSION_THRESHOLD = 0.013
        
        for i in range(1, len(candidates_step2_radar)):
            prev_slice = candidates_step2_radar[i-1]
            curr_slice = candidates_step2_radar[i]
        
            diff = curr_slice['radial_max'] - prev_slice['radial_max']
            
            if diff > PROTRUSION_THRESHOLD:
                print(f"    [Filter] Chin protrusion detected at h={curr_slice['h']:.4f} (Diff: {diff:.4f}). Excluding upper candidates.")
                cutoff_index = i
                break
        
        # 턱이 감지된 경우, 해당 단면부터 그 위쪽은 모두 후보에서 제외
        if cutoff_index != -1:
            candidates_step2_radar = candidates_step2_radar[:cutoff_index]

    # 최적 높이 선정 
    if len(candidates_step2_radar) > 0:
        print(f"    [Selection] Priority 1: Radar Passed & Protrusion Filtered ({len(candidates_step2_radar)} found).")
        best_candidate = min(candidates_step2_radar, key=lambda x: x['radius'])
        final_cut_height = best_candidate['h']
        
    elif len(candidates_step1_pca) > 0:
        print(f"    [Selection] Priority 2: Radar failed. Fallback to PCA candidates ({len(candidates_step1_pca)} found).")
        best_candidate = min(candidates_step1_pca, key=lambda x: x['radius'])
        final_cut_height = best_candidate['h']
        
    else:
        print(f"    [Warning] No valid candidates. Using Safe Fallback Height.")
        safe_fallback = min(fallback_height, chin_bottom_limit - 0.01)
        final_cut_height = safe_fallback

    # 그래프 탐색을 통한 선택적 헤드 제거 
    verts = body_mesh.vertices
    seed_mask = (verts[:, 1] > final_cut_height) & \
                (verts[:, 1] < final_cut_height + 0.02) & \
                ((verts[:, 0]**2 + verts[:, 2]**2) < 0.06**2)
    
    seed_indices = np.where(seed_mask)[0]
    
    if len(seed_indices) > 0:
        remove_mask = np.zeros(len(verts), dtype=bool)
        remove_mask[seed_indices] = True
        neighbors = body_mesh.vertex_neighbors
        stack = list(seed_indices)
        while stack:
            curr_idx = stack.pop()
            valid_next = [n for n in neighbors[curr_idx] if not remove_mask[n] and verts[n, 1] > final_cut_height]
            for n in valid_next: remove_mask[n] = True; stack.append(n)
        
        faces_to_remove = remove_mask[body_mesh.faces].any(axis=1)
        body_mesh.update_faces(~faces_to_remove)
        body_mesh.remove_unreferenced_vertices()

        # 잔여 파편 제거 
        components = body_mesh.split(only_watertight=False)
        keepers = []
        for comp in components:
            if len(comp.vertices) > 2000: keepers.append(comp); continue
            center = comp.centroid
            if center[1] < final_cut_height: keepers.append(comp); continue
            if (center[0]**2 + center[2]**2) > 0.15**2: keepers.append(comp); continue 
            extents = comp.extents
            if np.any(extents < 0.001): keepers.append(comp); continue
            if (np.min(extents) / np.max(extents)) > 0.5: continue 
            keepers.append(comp)
        
        if len(keepers) < len(components):
            body_mesh = trimesh.util.concatenate(keepers)
    else:
        print("[Warning] Fallback slicing used.")
        mask_remove = (verts[:, 1] > final_cut_height)
        body_mesh.update_faces(~mask_remove[body_mesh.faces].any(axis=1))
        body_mesh.remove_unreferenced_vertices()
        
    #body_mesh.export(os.path.join(output_dir, f'3_body_cut_dynamic{suffix}.obj'))

    # =============================================================================
    # [Step 3] 얼굴 메시 처리 
    # =============================================================================
    print(" [Step 3] Face Processing: Aligning and slicing DECA head mesh...")

    l_eye = np.mean(deca_lmk[36:42], axis=0)
    r_eye = np.mean(deca_lmk[42:48], axis=0)
    pivot = (l_eye + r_eye) / 2.0
    head_mesh.vertices -= pivot
    deca_lmk -= pivot

    verts_base = head_mesh.vertices.copy()
    mesh_min_y = np.min(verts_base[:, 1])
    mesh_chin_y = deca_lmk[8, 1]
    y_steps = np.linspace(mesh_min_y + 0.005, mesh_chin_y, 50)

    best_fallback = None
    min_circularity = float('inf')

    for y in y_steps:
        mask = (verts_base[:, 1] > y - 0.002) & (verts_base[:, 1] < y + 0.002)
        pts = verts_base[mask]
        if len(pts) > 10:
            x_rng = np.ptp(pts[:, 0]); z_rng = np.ptp(pts[:, 2])
            if x_rng > 0:
                ratio = z_rng / x_rng
                circularity = abs(1.0 - ratio)
                current_radius = (x_rng + z_rng) / 4.0
                center_xz = np.array([np.mean(pts[:,0]), np.mean(pts[:,2])])
                current_info = {'radius': current_radius, 'y': y, 'center': center_xz, 'circularity': circularity}
                if circularity < min_circularity:
                    min_circularity = circularity
                    best_fallback = current_info

    if best_fallback is None:
        final_y = mesh_min_y + 0.001
    else:
        final_y = best_fallback['y']
    
    head_mesh = head_mesh.slice_plane(plane_origin=[0, final_y, 0], plane_normal=[0, 1, 0])
    
    # [Method 1] 경계선 변수명: face_bottom_indices
    face_boundary_loop = find_boundary_loop_clean_py(head_mesh, [0, final_y, 0], [0, -1, 0])
    face_bottom_indices = []
    if face_boundary_loop is not None:
        face_bottom_indices = face_boundary_loop

    #head_mesh.export(os.path.join(output_dir, f'debug_4c_face_extruded{suffix}.obj'))

    # =============================================================================
    # [Step 4] 병합 및 토폴로지 연결 
    # =============================================================================
    print(" [Step 4] Mesh Merging: Synchronizing local coordinate systems (Body <-> Face)...")

    # [Method 1] 경계선 변수명: body_boundary_loop
    body_boundary_loop = find_boundary_loop_clean_py(body_mesh, [0, final_cut_height, 0], [0, 1, 0])
    
    if body_boundary_loop is not None and len(face_bottom_indices) > 0:

        # 1) 좌표계 동기화 (Method 1)
        if len(joints) >= 25:
            # Scaling
            b_l_eye_pos = joints[23]
            b_r_eye_pos = joints[24]
            body_eye_dist = np.linalg.norm(b_r_eye_pos - b_l_eye_pos)

            d_l_eye_pos = np.mean(deca_lmk[36:42], axis=0)
            d_r_eye_pos = np.mean(deca_lmk[42:48], axis=0)
            deca_eye_dist = np.linalg.norm(d_r_eye_pos - d_l_eye_pos)

            if deca_eye_dist > 0:
                scale_factor = body_eye_dist / deca_eye_dist
                print(f"    [Scaling] Applying scale factor: {scale_factor:.4f}")
                head_mesh.vertices *= scale_factor
                deca_lmk *= scale_factor
            else:
                print("    [Warning] DECA eye distance is zero. Skipping scaling.")

            # Body Frame
            b_l_eye = joints[23]
            b_r_eye = joints[24]
            b_neck = joints[NECK_IDX] 
            b_head = joints[HEAD_IDX] 
            b_vec_x = b_r_eye - b_l_eye
            b_vec_x /= np.linalg.norm(b_vec_x)
            b_vec_y_raw = b_head - b_neck 
            b_vec_y_raw /= np.linalg.norm(b_vec_y_raw)
            b_vec_z = np.cross(b_vec_x, b_vec_y_raw)
            b_vec_z /= np.linalg.norm(b_vec_z)
            b_vec_z = -b_vec_z 
            b_vec_x = -b_vec_x
            b_vec_y = np.cross(b_vec_z, b_vec_x)
            b_vec_y /= np.linalg.norm(b_vec_y)
            mat_body = np.stack([b_vec_x, b_vec_y, b_vec_z], axis=1)
            
            # Face Frame
            d_l_eye = np.mean(deca_lmk[36:42], axis=0)
            d_r_eye = np.mean(deca_lmk[42:48], axis=0)
            d_pivot = (d_l_eye + d_r_eye) / 2.0
            
            d_vec_x = d_r_eye - d_l_eye
            d_vec_x /= np.linalg.norm(d_vec_x)
            d_vec_y_raw = np.array([0.0, 1.0, 0.0])
            d_vec_z = np.cross(d_vec_x, d_vec_y_raw)
            d_vec_z /= np.linalg.norm(d_vec_z)
            d_vec_y = np.cross(d_vec_z, d_vec_x)
            d_vec_y /= np.linalg.norm(d_vec_y)
            mat_deca = np.stack([d_vec_x, d_vec_y, d_vec_z], axis=1)

            rot_matrix = np.dot(mat_body, mat_deca.T)
            head_mesh.vertices = np.dot(head_mesh.vertices, rot_matrix.T)
            print(f"    [Alignment] Corrected 180-deg Flip & 45-deg Tilt using Spine Vector.")

        # 2) 위치 정렬 (Method 1: Bounding Box Center 기반)
        b_verts = body_mesh.vertices[body_boundary_loop]
        f_verts = head_mesh.vertices[face_bottom_indices]

        def get_bbox_center_xz(pts):
            # 점들의 기하학적 중심(Bounding Box Center)을 반환
            min_xyz = np.min(pts, axis=0)
            max_xyz = np.max(pts, axis=0)
            center_x = (min_xyz[0] + max_xyz[0]) / 2.0
            center_z = (min_xyz[2] + max_xyz[2]) / 2.0
            return np.array([center_x, center_z])

        body_neck_center_xz = get_bbox_center_xz(b_verts)
        face_center_xz = get_bbox_center_xz(f_verts)
        
        # Body의 기하학적 중심점을 타겟 위치로 설정 
        target_pos = np.array([body_neck_center_xz[0], final_cut_height, body_neck_center_xz[1]])
        
        # Face의 현재 중심점 계산 
        face_mean_y = np.mean(f_verts[:, 1])
        face_current_center = np.array([face_center_xz[0], face_mean_y, face_center_xz[1]])
        
        # 두 중심점이 겹치도록 이동
        translation = target_pos - face_current_center
        head_mesh.vertices += translation 

        #head_mesh.export(os.path.join(output_dir, f'4_debug_aligned_face_only{suffix}.obj'))
        #trimesh.Scene([body_mesh, head_mesh]).export(os.path.join(output_dir, f'4_debug_alignment_scene{suffix}.obj'))

        # =============================================================================
        # [Step 5] Harmonic Warping 
        # =============================================================================
        print(" [Step 5] Harmonic Warping (Wide Range) - Switching to Method 2 Logic...")
        
        head_loop_verts = head_mesh.vertices[face_bottom_indices] 
        body_loop_verts = body_mesh.vertices[body_boundary_loop]
        
        # 1. 경계선 스냅
        tree_border = KDTree(head_loop_verts)
        _, indices_snap = tree_border.query(body_loop_verts)
        body_mesh.vertices[body_boundary_loop] = head_loop_verts[indices_snap]
        
        # 2. 하모닉 필드 계산 (범위 확장)
        tree_full = KDTree(head_loop_verts)
        dists, indices = tree_full.query(body_mesh.vertices)
        nearest_pts = head_loop_verts[indices]
        
        # 파라미터 튜닝 (Wide)
        sigma = 0.15
        weights = np.exp(-(dists**2) / (2 * sigma**2))
        height_limit_warp = final_cut_height - 0.05
        dist_from_axis_sq = (body_mesh.vertices[:, 0] - target_pos[0])**2 + (body_mesh.vertices[:, 2] - target_pos[2])**2
        radius_limit_sq = 0.12**2 

        mask_protect = (body_mesh.vertices[:, 1] < height_limit_warp) | (dist_from_axis_sq > radius_limit_sq)
        weights[mask_protect] = 0.0
        weights[body_boundary_loop] = 1.0 
        
        body_mesh.vertices = (1 - weights[:, np.newaxis]) * body_mesh.vertices + weights[:, np.newaxis] * nearest_pts
        
        # Debug Export Post-Warp
        debug_verts = np.vstack([body_mesh.vertices, head_mesh.vertices])
        debug_faces = np.vstack([body_mesh.faces, head_mesh.faces + len(body_mesh.vertices)])
        trimesh.Trimesh(vertices=debug_verts, faces=debug_faces, process=False)
        
        # =============================================================================
        # [Step 6] Zipper Stitching (Topology Fix with Existing Function)
        # =============================================================================
        print(" [Step 6] Zipper Stitching...")
        idx_head_start = len(body_mesh.vertices)
        combined_verts = np.vstack([body_mesh.vertices, head_mesh.vertices])
        head_faces_shifted = head_mesh.faces + idx_head_start
        
        # 1. 버텍스 및 중심점 준비
        b_loop_verts = body_mesh.vertices[body_boundary_loop]
        h_loop_verts = head_mesh.vertices[face_bottom_indices]
        
        b_center = np.mean(b_loop_verts, axis=0)
        h_center = np.mean(h_loop_verts, axis=0)
        
        # [Topology Fix] 2. 기존 함수(check_winding_order_py)를 사용해 방향 검사
        # Normal을 [0, 1, 0]으로 주면 XZ 평면 기준(위에서 본 시점)의 회전값을 줍니다.
        # 양수(+)면 반시계(CCW), 음수(-)면 시계(CW) 방향입니다.
        b_wind_val = check_winding_order_py(b_loop_verts, b_center, normal=[0, 1, 0])
        h_wind_val = check_winding_order_py(h_loop_verts, h_center, normal=[0, 1, 0])
        
        b_ccw = b_wind_val > 0
        h_ccw = h_wind_val > 0
        
        print(f"    [Topology Check] Body: {'CCW' if b_ccw else 'CW'} ({b_wind_val:.4f}), Head: {'CCW' if h_ccw else 'CW'} ({h_wind_val:.4f})")
        

        # 3. 방향 불일치 시 Head 루프 반전(Reverse)
        if b_ccw != h_ccw:
            print("    [Fix] Winding mismatch detected. Reversing Head Loop indices...")
            face_bottom_indices = face_bottom_indices[::-1] # 배열 뒤집기
        
        # 4. 루프 데이터 갱신 (뒤집혔을 수 있으므로 다시 추출)
        head_loop_verts = head_mesh.vertices[face_bottom_indices]
        n_body = len(body_boundary_loop)
        n_head = len(face_bottom_indices)
        
        bridge_faces = []
        
        # 5. 시작점 정렬 (KDTree)
        tree_h = KDTree(head_loop_verts)
        _, start_idx_h = tree_h.query(body_mesh.vertices[body_boundary_loop][0])
        
        # Head 루프를 시작점에 맞춰 회전(Roll)
        head_loop_rolled = np.roll(face_bottom_indices, -start_idx_h)
        
        # 6. 지퍼 알고리즘 실행
        bi = 0; hi = 0
        while bi < n_body or hi < n_head:
            curr_b = body_boundary_loop[bi % n_body]
            curr_h = head_loop_rolled[hi % n_head] + idx_head_start
            next_b = body_boundary_loop[(bi + 1) % n_body]
            next_h = head_loop_rolled[(hi + 1) % n_head] + idx_head_start
            
            # 비율에 따라 삼각형 생성
            if (bi + 1) / n_body < (hi + 1) / n_head: 
                bridge_faces.append([curr_b, next_b, curr_h]); bi += 1
            else: 
                bridge_faces.append([curr_b, curr_h, next_h]); hi += 1
                
        all_faces = np.vstack([body_mesh.faces, head_faces_shifted, bridge_faces])
        merged_mesh = trimesh.Trimesh(vertices=combined_verts, faces=all_faces, process=False)
        merged_mesh.merge_vertices(merge_tex=True, merge_norm=True)

        # [Debug Export]
        #debug_zipper_name = f'debug_6_zipper_raw{suffix}.obj'
        #merged_mesh.export(os.path.join(output_dir, debug_zipper_name))
        #print(f"    -> [Debug Saved] Raw Stitched Mesh: {debug_zipper_name}")
        # =============================================================================
        # [Step 7] Zone-Based Bi-Laplacian Smoothing (Method 2 Logic)
        # =============================================================================
        print(" [Step 7] Applying Bi-Laplacian Smoothing...")
        
        # [Method 2 Logic] Seam height calculation using final_cut_height
        GAP_OFFSET = 0.005
        seam_height_for_smoothing = final_cut_height + (GAP_OFFSET / 2.0)
        
        merged_mesh.fix_normals()
        merged_mesh = smooth_neck_zone_based(
            merged_mesh,
            cut_height = seam_height_for_smoothing, 
            center_pos = target_pos
        )

        # =============================================================================
        # [Step 8] Save (Method 2 Logic)
        # =============================================================================
        print(" [Step 8] Saving...")
        merged_mesh.vertices = np.dot(merged_mesh.vertices, final_rot_mat)
        merged_mesh.vertices += neck_joint_orig
        rot_upright = R.from_euler('x', 180, degrees=True).as_matrix()
        merged_mesh.vertices = np.dot(merged_mesh.vertices, rot_upright.T)

        final_name = f'Method_B_{suffix}.obj'
        merged_mesh.export(os.path.join(output_dir, final_name))
        print(f"    -> Saved: {final_name}")
        
    else:
        print("[Error] Boundary loops not found or Face slice failed. Cannot proceed with stitching.")

# =============================================================================
# 실행 
# =============================================================================
if 'body_mesh_global' in globals() and body_mesh_global is not None:
    # 1. 원본 표정 처리
    if 'deca_mesh_raw_global' in globals():
        stitch_hybrid_final(
            deca_mesh_raw_global, deca_lmk_raw_global, 
            body_mesh_global, body_joints_global, 
            save_dir, suffix="_original"
        )
    
    # 2. 수정 표정 처리 
    if 'deca_mesh_mod_global' in globals():
        stitch_hybrid_final(
            deca_mesh_mod_global, deca_lmk_mod_global, 
            body_mesh_global, body_joints_global, 
            save_dir, suffix="_modified"
        )
else:
    print("[Error] Global variables missing.")


[Pipeline Start] Integrated Hybrid Solution

[Pipeline Start] Geometry Stitching Process: Original
 [Step 0] Preprocessing: Normalizing coordinates (Translating Neck to Global Origin)...
 [Step 1] Body Alignment: Compensating for spinal curvature (Y-axis alignment)...
 [Step 2] Adaptive Slicing: 1st PCA Filter & 2nd Radial Radar Check (Method 1 Logic)...
    [Warning] No valid candidates. Using Safe Fallback Height.
 [Step 3] Face Processing: Aligning and slicing DECA head mesh...
 [Step 4] Mesh Merging: Synchronizing local coordinate systems (Body <-> Face)...
    [Scaling] Applying scale factor: 0.9794
    [Alignment] Corrected 180-deg Flip & 45-deg Tilt using Spine Vector.
 [Step 5] Harmonic Warping (Wide Range) - Switching to Method 2 Logic...
 [Step 6] Zipper Stitching...
    [Topology Check] Body: CW (-0.0229), Head: CCW (0.0247)
    [Fix] Winding mismatch detected. Reversing Head Loop indices...
 [Step 7] Applying Bi-Laplacian Smoothing...
    -> Smoothing Zone: 500 vertices.
 

In [9]:
# 7. Method C: 이종 잠재 통합 및 파라미터 기반 합성

import torch
import numpy as np
import os
import trimesh
import smplx
import cv2
from scipy.spatial.transform import Rotation as R

print(f"\n" + "="*60)
print("[Pipeline Start] Method 3: Fusion (Auto-Align + Final X180 Only)")
print("="*60)

# =============================================================================
# [Step 1] 파라미터 준비
# =============================================================================
try:
    raw_jaw_global = codedict_raw['pose'][:, 3:6]
    raw_exp_global = codedict_raw['exp']
    mod_jaw_global = codedict_mod['pose'][:, 3:6]
    mod_exp_global = codedict_mod['exp']
    print(" [Step 1] Parameter Extraction: Ready.")
except NameError:
    print(" [Error] Dependency Missing: 'codedict_raw' not defined.")
    raise

# =============================================================================
# [Step 2] 데이터 정규화
# =============================================================================
print(" [Step 2] Normalizing Pose Data...")

body_pose_fixed = convert_to_axis_angle(flatten_tensor(body_params_global['body_pose']))
left_hand_fixed = convert_to_axis_angle(flatten_tensor(body_params_global['left_hand_pose']))
right_hand_fixed = convert_to_axis_angle(flatten_tensor(body_params_global['right_hand_pose']))
global_orient_fixed = convert_to_axis_angle(flatten_tensor(body_params_global['global_orient']))

# =============================================================================
# [Step 3] 모델 초기화
# =============================================================================
smplx_model_path = os.path.join(MODELS_DIR, 'smplx', 'models')
hand_dim = left_hand_fixed.shape[-1]
is_hand_pca = (hand_dim < 45)

print(f" [Check] Final Hand Dimension: {hand_dim} (PCA Mode: {is_hand_pca})")

fusion_layer = smplx.create(
    smplx_model_path, model_type='smplx', gender='neutral', 
    use_pca=is_hand_pca,      
    num_pca_comps=hand_dim,   
    flat_hand_mean=True,      
    num_betas=10, 
    num_expression_coeffs=50
).to(device)

# =============================================================================
# [Step 4] 결과 생성, SVD 정렬 
# =============================================================================
print(" [Step 4] Generating & Aligning via Skeleton...")

# 원본 관절 데이터 확보
if 'body_joints_global' not in globals():
    raise ValueError("[Error] 'body_joints_global' variable is missing. Run Method 1/2 preprocessing first.")

target_joints_ref = body_joints_global.copy()

rot_final_adjustment = R.from_euler('xy', [180, 0], degrees=True).as_matrix()
print("   [Info] Final Adjustment Rotation Configured: X=180, Y=0 (No Flip)")

# 파라미터 설정
body_kwargs = {
    'global_orient': global_orient_fixed,
    'body_pose': body_pose_fixed,
    'left_hand_pose': left_hand_fixed,
    'right_hand_pose': right_hand_fixed,
    'betas': body_params_global['betas'][:, :10].to(device),
    'transl': torch.zeros((1, 3)).to(device), # 정렬 함수가 위치를 잡으므로 0으로 초기화
    'return_verts': True
}

# --- Case A: Body + Raw Face ---
print(" -> Generating Fusion A (Raw Expression)...")
jaw_pose_fixed = ensure_axis_angle_single(raw_jaw_global)
expression_fixed = raw_exp_global[:, :50].to(device)

with torch.no_grad():
    output = fusion_layer(
        jaw_pose=jaw_pose_fixed,
        expression=expression_fixed,
        **body_kwargs
    )
    
verts_gen = output.vertices[0].detach().cpu().numpy()

# 1. SVD 정렬 
print("   [Aligning] Matching skeleton to original scan...")
verts_aligned = align_generated_to_original(verts_gen, output.joints, target_joints_ref)

verts_final_raw = np.dot(verts_aligned, rot_final_adjustment.T)

save_obj(trimesh.Trimesh(verts_final_raw, fusion_layer.faces), os.path.join(save_dir, 'Method_C_original.obj'))


# --- Case B: Body + Modified Face  ---
print(" -> Generating Fusion B (Modified Expression)...")
jaw_pose_mod_fixed = ensure_axis_angle_single(mod_jaw_global)
expression_mod_fixed = mod_exp_global[:, :50].to(device)

with torch.no_grad():
    output_mod = fusion_layer(
        jaw_pose=jaw_pose_mod_fixed,
        expression=expression_mod_fixed,
        **body_kwargs
    )

verts_mod_gen = output_mod.vertices[0].detach().cpu().numpy()

# 1. SVD 정렬 
print("   [Aligning] Matching skeleton to original scan...")
verts_mod_aligned = align_generated_to_original(verts_mod_gen, output_mod.joints, target_joints_ref)

verts_final_mod = np.dot(verts_mod_aligned, rot_final_adjustment.T)

save_obj(trimesh.Trimesh(verts_final_mod, fusion_layer.faces), os.path.join(save_dir, 'Method_C_Modified.obj'))

print("=== Method 3 Complete ===")


[Pipeline Start] Method 3: Fusion (Auto-Align + Final X180 Only)
 [Step 1] Parameter Extraction: Ready.
 [Step 2] Normalizing Pose Data...
   [Info] Converted RotMatrix to AxisAngle: 189 -> 63
   [Info] Converted RotMatrix to AxisAngle: 135 -> 45
   [Info] Converted RotMatrix to AxisAngle: 135 -> 45
 [Check] Final Hand Dimension: 45 (PCA Mode: False)
 [Step 4] Generating & Aligning via Skeleton...
   [Info] Final Adjustment Rotation Configured: X=180, Y=0 (No Flip)
 -> Generating Fusion A (Raw Expression)...
   [Aligning] Matching skeleton to original scan...
 -> Saved: Method_C_original.obj
 -> Generating Fusion B (Modified Expression)...
   [Aligning] Matching skeleton to original scan...
 -> Saved: Method_C_Modified.obj
=== Method 3 Complete ===


In [11]:
# 8. Method D: 신뢰도 가중 특징 융합

import os
import torch
import numpy as np
import trimesh
from pixielib.utils import util
from pixielib.utils import rotation_converter as converter

print(f"\n" + "="*80)
print("[Pipeline Start] Method D: Parameter Fusion (Smooth Mesh)")
print("Logic: PIXIE Body + DECA Expression/Jaw -> Integrated SMPL-X Mesh")
print("="*80)

def generate_fusion_mesh(suffix, deca_codedict):
    print(f"\n -> Generating Fusion Mesh for: {suffix}...")
    
    # 1. PIXIE Encode (Body Parameters)
    with torch.no_grad():
        batch = testdata[0]
        util.move_dict_to_device(batch, 'cpu')
        batch["image"] = batch["image"].unsqueeze(0)
        batch["image_hd"] = batch["image_hd"].unsqueeze(0)
        data = {"body": batch}
        param_dict = pixie_model.encode(data, threthold=False, keep_local=True, copy_and_paste=False)
        pixie_input = param_dict["body"]

    # 2. DECA Parameter Injection (Expression & Jaw)
    deca_exp = deca_codedict['exp'].cpu() if isinstance(deca_codedict['exp'], torch.Tensor) else torch.from_numpy(deca_codedict['exp'])
    deca_pose = deca_codedict['pose'].cpu() if isinstance(deca_codedict['pose'], torch.Tensor) else torch.from_numpy(deca_codedict['pose'])
    
    # Exp 차원 맞춤
    n_exp = pixie_cfg.model.n_exp
    if deca_exp.shape[1] >= n_exp: 
        pixie_input["exp"] = deca_exp[:, :n_exp]
    else: 
        pixie_input["exp"] = torch.nn.functional.pad(deca_exp, (0, n_exp - deca_exp.shape[1]))
        
    # Jaw Pose 변환 (Axis-Angle -> Euler)
    jaw_axis = deca_pose[0, 3:6].reshape(1, 3)
    jaw_rotmat = converter.batch_axis2matrix(jaw_axis)
    jaw_euler = converter._compute_euler_from_matrix(jaw_rotmat, seq='xyz', extrinsic=False)
    pixie_input["jaw_pose"] = jaw_euler

    # 3. PIXIE Decode (Integrated Mesh 생성)
    with torch.no_grad():
        pred = pixie_model.decode(pixie_input, param_type="body")
    verts = pred["vertices"][0].detach().cpu().numpy()
    faces = pixie_model.smplx.faces_tensor.detach().cpu().numpy()

    # 4. Save (디테일 적용 없이 저장)
    verts[:, 1] *= -1; verts[:, 2] *= -1
    final_name = f'Method_D{suffix}.obj'
    try:
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(os.path.join(save_dir, final_name))
        print(f" -> [Success] Saved: {final_name} (Smooth Fusion)")
    except Exception as e: print(f" -> [Error] Save failed: {e}")

# 실행
if 'codedict_raw' in globals() and 'codedict_mod' in globals():
    generate_fusion_mesh("_original", codedict_raw)
    generate_fusion_mesh("_modified", codedict_mod)
else:
    print("[Error] codedict 데이터가 없습니다.")


[Pipeline Start] Method D: Parameter Fusion (Smooth Mesh)
Logic: PIXIE Body + DECA Expression/Jaw -> Integrated SMPL-X Mesh

 -> Generating Fusion Mesh for: _original...
 -> [Success] Saved: Method_D_original.obj (Smooth Fusion)

 -> Generating Fusion Mesh for: _modified...
 -> [Success] Saved: Method_D_modified.obj (Smooth Fusion)


In [12]:
#9. Method E: 키포인트 재투영 및 시간 일관성 피팅

import os
import torch
import numpy as np
import trimesh
import copy
from pixielib.utils import util
from pixielib.utils import rotation_converter as converter
from pixielib.models.FLAME import texture_flame2smplx 

print(f"\n" + "="*80)
print("[Pipeline Start] Method 4: Detail Transfer (Strict Reproduction)")
print("Logic: SVAD/DECA Detail Transfer Pipeline with Masking & Euler Conversion")
print("="*80)

# =============================================================================
# [설정] 경로 및 데이터 로드 
# =============================================================================
# PIXIE 데이터 폴더 내의 필수 파일 경로 설정
TOPO_OBJ_PATH = os.path.join(PIXIE_DATA_DIR, 'SMPL_X_template_flame_uv.obj') 
CACHED_NPY_PATH = os.path.join(PIXIE_DATA_DIR, 'flame2smplx_cached_code.npy')

# 파일이 없으면 config에 설정된 기본값 시도
if not os.path.exists(TOPO_OBJ_PATH): TOPO_OBJ_PATH = pixie_cfg.model.topology_path
if not os.path.exists(CACHED_NPY_PATH): CACHED_NPY_PATH = pixie_cfg.model.flame2smplx_cached_path

print(f" [Check] Topology Path: {TOPO_OBJ_PATH}")
print(f" [Check] Cached NPY Path: {CACHED_NPY_PATH}")

if not os.path.exists(TOPO_OBJ_PATH) or not os.path.exists(CACHED_NPY_PATH):
    print("[Error] 필수 파일을 찾을 수 없습니다. PIXIE Data 폴더를 확인하세요.")
else:
    # 1. Cached Data 로드
    cached_data = np.load(CACHED_NPY_PATH, allow_pickle=True, encoding="latin1").item()
    target_res = int(cached_data.get("target_resolution", 1024))
    
    # 2. Topology 로드 및 UV 매핑 계산
    topo_verts_t, topo_uvcoords_t, topo_faces_t, topo_uvfaces_t = util.load_obj(TOPO_OBJ_PATH)
    topo_uvcoords = topo_uvcoords_t.detach().cpu().numpy().astype(np.float32)
    topo_faces = topo_faces_t.detach().cpu().numpy().astype(np.int32)
    topo_uvfaces = topo_uvfaces_t.detach().cpu().numpy().astype(np.int32)

    V_count = int(topo_verts_t.shape[0])
    uv_per_v = np.zeros((V_count, 2), dtype=np.float32)
    uv_cnt = np.zeros((V_count,), dtype=np.int32)

    for f in range(topo_faces.shape[0]):
        for k in range(3):
            vi, uvi = topo_faces[f, k], topo_uvfaces[f, k]
            if 0 <= vi < V_count and 0 <= uvi < topo_uvcoords.shape[0]:
                uv_per_v[vi] += topo_uvcoords[uvi]
                uv_cnt[vi] += 1
    mask = uv_cnt > 0
    uv_per_v[mask] /= uv_cnt[mask][:, None]
    print(" [Init] UV Mapping Prepared.")


def generate_detailed_mesh(suffix, face_image_path, deca_codedict):
    print(f"\n -> Generating Detailed Mesh for: {suffix}...")
    
    # 1. PIXIE Encode 
    with torch.no_grad():
        batch = testdata[0] 
        util.move_dict_to_device(batch, 'cpu')
        batch["image"] = batch["image"].unsqueeze(0)
        batch["image_hd"] = batch["image_hd"].unsqueeze(0)
        data = {"body": batch}
        param_dict = pixie_model.encode(data, threthold=False, keep_local=True, copy_and_paste=False)
        pixie_input = param_dict["body"]

    # 2. Parameter Injection
    # DECA 파라미터 준비
    deca_exp = deca_codedict['exp'].cpu() if isinstance(deca_codedict['exp'], torch.Tensor) else torch.from_numpy(deca_codedict['exp'])
    deca_pose = deca_codedict['pose'].cpu() if isinstance(deca_codedict['pose'], torch.Tensor) else torch.from_numpy(deca_codedict['pose'])
    
    # Exp 주입 
    n_exp = pixie_cfg.model.n_exp
    if deca_exp.shape[1] >= n_exp: pixie_input["exp"] = deca_exp[:, :n_exp]
    else: pixie_input["exp"] = torch.nn.functional.pad(deca_exp, (0, n_exp - deca_exp.shape[1]))
    
    # Jaw Pose 변환
    jaw_axis = deca_pose[0, 3:6].reshape(1, 3)
    jaw_rotmat = converter.batch_axis2matrix(jaw_axis)
    jaw_euler = converter._compute_euler_from_matrix(jaw_rotmat, seq='xyz', extrinsic=False)
    pixie_input["jaw_pose"] = jaw_euler

    # 3. PIXIE Decode
    with torch.no_grad():
        pred = pixie_model.decode(pixie_input, param_type="body")
    verts = pred["vertices"][0].detach().cpu().numpy()
    faces = pixie_model.smplx.faces_tensor.detach().cpu().numpy()

    # 4. Detail Transfer 
    print("    [Detail] Calculating & Applying Displacement...")
    # 4-1. Extract Displacement from Image (with Masks)
    flame_disp_hw1 = _get_deca_displacement_map_from_path(face_image_path)
    
    # 4-2. Convert FLAME UV -> SMPL-X UV
    smplx_disp_canvas = np.zeros((target_res, target_res, 1), dtype=np.float32)
    smplx_disp_hw1 = texture_flame2smplx(cached_data, flame_disp_hw1, smplx_disp_canvas)

    # 4-3. Apply to Vertices
    if verts.shape[0] == uv_per_v.shape[0]:
        disp_v = _bilinear_sample_hw1(smplx_disp_hw1, uv_per_v)
        N_v = _vertex_normals(verts, faces)
        DISP_SCALE = 1.0 
        verts = verts + (N_v * disp_v[:, None]) * DISP_SCALE
    else:
        print("    [Warn] Vertex count mismatch. Skipping displacement.")

    # 5. Save
    verts[:, 1] *= -1; verts[:, 2] *= -1
    
    final_name = f'Method_E{suffix}.obj'
    try:
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(os.path.join(save_dir, final_name))
        print(f" -> [Success] Saved: {final_name}")
    except Exception as e: print(f" -> [Error] Save failed: {e}")

# =============================================================================
# 실행
# =============================================================================
if 'codedict_raw' in globals() and 'codedict_mod' in globals():
    # Case 1: Original Expression
    generate_detailed_mesh("_original", SOURCE_FACE_IMAGE_PATH, codedict_raw)
    
    # Case 2: Modified Expression
    generate_detailed_mesh("_modified", SOURCE_FACE_IMAGE_PATH, codedict_mod)
else:
    print("[Error] 필수 데이터(codedict_raw/mod) 누락. 이전 셀을 실행하세요.")

In [13]:
# 10. 성능지표

import os
import time
import math
import numpy as np
import pandas as pd
from PIL import Image

# -----------------------------
# [사용자 입력]
# -----------------------------

# 1. 원본 이미지 경로 (Ground Truth)
BODY_IMAGE_PATH = str(INPUT_DIR / "SMPL-X" / "body5.jpg")

# 2. 비교할 결과물 OBJ 파일 경로 리스트 (여러 개 입력 가능)
# 예: [r"C:\Results\output1.obj", r"C:\Results\output2.obj"]
TARGET_OBJ_PATHS = [
    str(PROJECT_ROOT / "result_output_jupyter" / "Method_A_original.obj"),
    str(PROJECT_ROOT / "result_output_jupyter" / "Method_B__original.obj"),
    str(PROJECT_ROOT / "result_output_jupyter" / "Method_C_original.obj"),
    str(PROJECT_ROOT / "result_output_jupyter" / "Method_D_original.obj"),
    str(PROJECT_ROOT / "result_output_jupyter" / "Method_E_original.obj")
]
# 3. 평가 결과를 저장할 폴더 경로
OUTPUT_DIR = r"result_output_jupyter"

# -----------------------------
# 1. 라이브러리 체크 및 로드
# -----------------------------
try:
    import torch
    import torch.nn.functional as F
    from torchvision import transforms, models
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    print("[WARN] torch/torchvision import 실패 -> LPIPS/COSINE 사용 불가:", e)

try:
    import lpips
    LPIPS_OK = True
except Exception as e:
    LPIPS_OK = False
    print("[WARN] lpips import 실패 -> LPIPS 사용 불가:", e)

try:
    import trimesh
    import pyrender
except Exception as e:
    raise ImportError("렌더링을 위해 trimesh/pyrender가 필요합니다. (pip install trimesh pyrender pyglet)")

# -----------------------------
# 2. 유틸리티 함수 정의
# -----------------------------
def _imread_rgb(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"이미지 파일을 찾을 수 없습니다: {path}")
    img = Image.open(path).convert("RGB")
    return np.array(img)

def _mad_mean(a, b, mask2d=None):
    da = a.astype(np.float32) / 255.0
    db = b.astype(np.float32) / 255.0
    diff = np.abs(da - db)
    if mask2d is not None:
        m = mask2d.astype(bool)
        diff = diff[m]
    return float(np.mean(diff)) if diff.size else float("nan")

def _bbox_from_mask(mask, pad=10):
    H, W = mask.shape[:2]
    ys, xs = np.where(mask > 0)
    if len(xs) == 0: return None
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
    x1 = min(W-1, x1 + pad); y1 = min(H-1, y1 + pad)
    return (x0, y0, x1, y1)

def _crop(arr, bbox):
    x0, y0, x1, y1 = bbox
    return arr[y0:y1+1, x0:x1+1]

# SSIM (수정된 버전)
def _try_import_ssim():
    try:
        from skimage.metrics import structural_similarity as ssim
        def _ssim_fn(img1, img2):
            # 이미지의 최소 변 길이가 7보다 작으면 win_size를 줄임
            min_side = min(img1.shape[0], img1.shape[1])
            win_size = min(7, min_side)
            
            # win_size는 반드시 홀수여야 함
            if win_size % 2 == 0:
                win_size -= 1
            
            # 너무 작아서 계산 불가할 경우 처리
            if win_size < 3:
                return float("nan")

            try:
                # 최신 버전 (scikit-image >= 0.19)
                return float(ssim(img1, img2, channel_axis=2, data_range=255, win_size=win_size))
            except TypeError:
                # 구 버전 (scikit-image < 0.19)
                return float(ssim(img1, img2, multichannel=True, data_range=255, win_size=win_size))
                
        return ("skimage", _ssim_fn)
    except ImportError:
        return (None, None)

SSIM_BACKEND, SSIM_FN = _try_import_ssim()

# PSNR
def _psnr_masked(real_uint8, fake_uint8, mask_bool):
    r = real_uint8.astype(np.float32); f = fake_uint8.astype(np.float32)
    m = mask_bool.astype(bool)
    r = r[m]; f = f[m]
    if r.size == 0: return float("nan")
    mse = float(np.mean((r - f) ** 2))
    if mse == 0: return 100.0
    return float(10.0 * np.log10((255.0 ** 2) / mse))

# -----------------------------
# 3. 모델 초기화 (LPIPS / VGG - CPU 강제)
# -----------------------------
DEVICE = "cpu"
LPIPS_MODEL = None
VGG_MODEL = None
COS_FN = None

if TORCH_OK:
    device = torch.device(DEVICE)
    # LPIPS
    if LPIPS_OK:
        try:
            LPIPS_MODEL = lpips.LPIPS(net="alex").to(device).eval()
        except: pass
    
    # VGG16 (Cosine)
    try:
        try:
            from torchvision.models import VGG16_Weights
            VGG_MODEL = models.vgg16(weights=VGG16_Weights.DEFAULT).to(device).eval()
        except:
            VGG_MODEL = models.vgg16(pretrained=True).to(device).eval()
        from torch.nn import CosineSimilarity
        COS_FN = CosineSimilarity(dim=1, eps=1e-6)
    except Exception as e:
        print("[WARN] VGG16 로드 실패:", e)

def _lpips_score(real_uint8, fake_uint8, mask_bool):
    if (not TORCH_OK) or (LPIPS_MODEL is None): return float("nan")
    rc = real_uint8.copy(); fc = fake_uint8.copy()
    m = mask_bool.astype(bool)
    rc[~m] = 0; fc[~m] = 0
    t1 = torch.from_numpy(rc).permute(2,0,1).unsqueeze(0).float() / 255.0
    t2 = torch.from_numpy(fc).permute(2,0,1).unsqueeze(0).float() / 255.0
    t1 = t1 * 2 - 1; t2 = t2 * 2 - 1
    t1 = F.interpolate(t1, size=(256,256), mode="bilinear", align_corners=False)
    t2 = F.interpolate(t2, size=(256,256), mode="bilinear", align_corners=False)
    with torch.no_grad():
        v = LPIPS_MODEL(t1.to(device), t2.to(device)).item()
    return float(v)

def _cosine_vgg(real_uint8, fake_uint8, mask_bool):
    if (not TORCH_OK) or (VGG_MODEL is None) or (COS_FN is None): return float("nan")
    rc = real_uint8.copy(); fc = fake_uint8.copy()
    m = mask_bool.astype(bool)
    rc[~m] = 0; fc[~m] = 0
    tfm = transforms.Compose([
        transforms.ToPILImage(), transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    t1 = tfm(rc).unsqueeze(0).to(device)
    t2 = tfm(fc).unsqueeze(0).to(device)
    with torch.no_grad():
        f1 = VGG_MODEL.features(t1).flatten(1)
        f2 = VGG_MODEL.features(t2).flatten(1)
        sim = COS_FN(f1, f2).item()
    return float(sim)

# -----------------------------
# 4. 렌더링 함수
# -----------------------------
def _look_at(eye, target, up=np.array([0,1,0], dtype=np.float32)):
    z = eye - target; z /= (np.linalg.norm(z) + 1e-9)
    x = np.cross(up, z); x /= (np.linalg.norm(x) + 1e-9)
    y = np.cross(z, x)
    T = np.eye(4, dtype=np.float32)
    T[0,:3] = x; T[1,:3] = y; T[2,:3] = z
    T[:3,3] = eye
    return T

def render_obj_to_rgb_depth(obj_path, W, H):
    if not os.path.exists(obj_path):
        raise FileNotFoundError(f"OBJ 파일을 찾을 수 없습니다: {obj_path}")

    mesh = trimesh.load(obj_path, process=False)
    if isinstance(mesh, trimesh.Scene):
        mesh = trimesh.util.concatenate(list(mesh.geometry.values()))

    bbox = mesh.bounds
    center = bbox.mean(axis=0)
    ext = (bbox[1] - bbox[0])
    radius = float(np.linalg.norm(ext) + 1e-9)

    scene = pyrender.Scene(bg_color=[0,0,0,0], ambient_light=[0.3,0.3,0.3])
    pm = pyrender.Mesh.from_trimesh(mesh, smooth=False)
    scene.add(pm)

    cam_dist = max(1.0, radius * 1.5)
    eye = center + np.array([0, 0, cam_dist], dtype=np.float32)
    cam_pose = _look_at(eye, center, up=[0, 1, 0])

    camera = pyrender.PerspectiveCamera(yfov=np.pi/3.0)
    scene.add(camera, pose=cam_pose)

    light = pyrender.DirectionalLight(color=np.ones(3), intensity=2.0)
    scene.add(light, pose=cam_pose)

    r = pyrender.OffscreenRenderer(viewport_width=W, viewport_height=H)
    color, depth = r.render(scene, flags=pyrender.RenderFlags.RGBA)
    r.delete()

    return color[...,:3].astype(np.uint8), (depth > 0)

# -----------------------------
# 5. 메인 실행 (Main Execution)
# -----------------------------
if __name__ == "__main__":
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print(f"[Info] 결과 저장 폴더 생성됨: {OUTPUT_DIR}")

    print(f"[Start] Metrics Evaluation")
    print(f" -> Image: {os.path.basename(BODY_IMAGE_PATH)}")
    print(f" -> Targets: {len(TARGET_OBJ_PATHS)} files")

    results = []

    try:
        # 1. Real Image Load (한 번만 로드)
        real = _imread_rgb(BODY_IMAGE_PATH)
        H, W = real.shape[:2]

        for obj_path in TARGET_OBJ_PATHS:
            key = os.path.basename(obj_path)
            t0 = time.perf_counter()

            # 2. Fake Image Render
            try:
                fake, mask = render_obj_to_rgb_depth(obj_path, W, H)
            except Exception as e:
                print(f" -> Render Error ({key}): {e}")
                continue

            # 3. Crop
            bbox = _bbox_from_mask(mask.astype(np.uint8))
            if bbox is None:
                print(f" -> Empty Render ({key})")
                continue

            real_c = _crop(real, bbox)
            fake_c = _crop(fake, bbox)
            mask_c = _crop(mask.astype(np.uint8), bbox).astype(bool)

            # 4. Calculate Metrics
            l1 = _mad_mean(real_c, fake_c, mask2d=mask_c)
            
            ssim_val = float("nan")
            if SSIM_FN is not None:
                rc = real_c.copy(); fc = fake_c.copy()
                rc[~mask_c] = 0; fc[~mask_c] = 0
                try: ssim_val = float(SSIM_FN(rc, fc))
                except: pass

            psnr_val = _psnr_masked(real_c, fake_c, mask_c)
            lpips_val = _lpips_score(real_c, fake_c, mask_c)
            cos_val = _cosine_vgg(real_c, fake_c, mask_c)

            dt = time.perf_counter() - t0
            
            print(f" -> Checked: {key} (L1: {l1:.4f}, Time: {dt:.2f}s)")
            
            results.append({
                "Filename": key,
                "L1": l1,
                "SSIM": ssim_val,
                "PSNR": psnr_val,
                "LPIPS": lpips_val,
                "COSINE": cos_val,
                "Time(s)": dt
            })

        # 5. 결과 저장
        if len(results) > 0:
            df_res = pd.DataFrame(results)
            print("\n" + "="*40)
            print(" [Performance Metrics Result] ")
            print("="*40)
            display(df_res)
            
            save_path = os.path.join(OUTPUT_DIR, "metrics_summary.csv")
            df_res.to_csv(save_path, index=False)
            print(f"\n[Done] 결과가 저장되었습니다: {save_path}")
        else:
            print("\n[Warning] 유효한 결과가 없습니다.")

    except Exception as e:
        print(f"\n[Fatal Error] 실행 중 오류 발생: {e}")

,Filename,L1,SSIM,PSNR,LPIPS,COSINE,Time(s)
0,Method_A_original.obj,0.327005,0.868867,8.602925,0.181008,0.622865,2.157765
1,Method_B__original.obj,0.326778,0.868876,8.606405,0.180893,0.630173,1.260893
2,Method_C_original.obj,0.326162,0.869660,8.621207,0.175424,0.606792,1.233292
3,Method_D_original.obj,0.326825,0.869475,8.608592,0.181167,0.632748,1.206395
4,Method_E_original.obj,0.326827,0.869471,8.608532,0.181161,0.632853,1.264064



[Done] 결과가 저장되었습니다: result_output_jupyter\metrics_summary.csv
